# Dependence Modelling in Multi-Asset Derivatives Pricing

Empirical pipeline for the MSc thesis *Dependence Modelling in Multi-Asset Derivatives Pricing: An Empirical Analysis of Linear Correlation versus Copula-Based Approaches for Equity-FX Basket Option Pricing* (Mariglen Tanushi, Università Cattolica del Sacro Cuore, MSc Banking and Finance, A.Y. 2025/2026; supervisor Prof. Paolo Gualtieri).

The notebook builds option-implied marginal distributions for the S&P 500 and EUR/USD, calibrates five copula families on daily returns and prices 50/50 basket calls by Monte Carlo, so that the effect of the dependence structure on the price can be isolated.

## Design

| Item | Choice |
|------|--------|
| SPX option chains | optionsDX end-of-day archives (Cboe-derived) |
| EUR/USD volatility | LSEG REFINITIV OTC composite (ATM, 25Δ and 10Δ risk reversals and butterflies) |
| EUR/USD spot | LSEG REFINITIV (EUR=) |
| USD rate r | FRED DGS3MO (3-month Treasury constant maturity) |
| EUR rate r_f | ECB Data Portal, 3-month Euribor monthly average; each date uses the latest average complete on that date |
| SPX dividend yield q | Put-call parity on near-the-money SPX pairs (Hull 2022, §17.4) |
| VIX | Cboe official daily close |
| Sample window | 2018-01-01 to 2022-12-31 |
| Snapshots | 74 targets (60 month-ends + 14 stress-event dates); 72 with both surfaces |
| Maturities | 0.25 / 0.50 / 1.00 / 2.00 years |
| Strikes | K/F_B = 0.90 / 0.95 / 1.00 / 1.05 / 1.10, with F_B = 0.5 e^{(r-q)T} + 0.5 e^{(r-r_f)T} |
| Copulas | Gaussian, Frank, Clayton, Gumbel and Student-t (canonical maximum likelihood); survival, BB7 and SJC families, GARCH filtering and Cramér-von Mises tests as robustness |
| Regimes | Calm: VIX < 20; Elevated: 20 ≤ VIX < 30; Stress: VIX ≥ 30 (snapshots by same-day close, daily returns by previous-day close; terciles as robustness) |
| Marginals | SVI per maturity with butterfly, calendar and Lee-bound penalties; Breeden-Litzenberger CDF; flat-smile lognormal benchmark |
| Monte Carlo | 100,000 draws per price, antithetic sampling on the driving uniforms (Glasserman 2003, §4.2), common random numbers across copulas, standard error of every copula-minus-Gaussian difference |

## Structure

| Section | Content |
|---|---|
| 1 | Setup, paths and snapshot dates |
| 2 | SPX option chains: extraction, filters, rates and dividend yield, implied volatility inversion |
| 3 | SPX volatility surfaces |
| 4 | VIX |
| 5 | Public market data |
| 6 | EUR/USD spot and paired daily returns |
| 7 | EUR/USD volatility surfaces |
| 8 | Copula calibration and robustness |
| 9 | Option-implied marginals, Monte Carlo pricing and pricing checks |
| 10 | Tables, figures and `results_full.json` |

## Outputs

Section 10 writes to `outputs/` inside the project folder:
- `results_full.json`: every number reported in the thesis
- `tables/`: result tables (CSV and JSON)
- `figures/`: the thesis figures


## Section 1. Setup, paths, snapshot dates


In [ ]:
# ============================================================
# Cell 1.1 - Install packages, mount Google Drive, define paths
# ============================================================
!pip install py7zr yfinance pandas_datareader requests arch -q

import os, py7zr, json, math, warnings, shutil, requests
from io import StringIO
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
from scipy.optimize import brentq, minimize_scalar
from scipy.interpolate import griddata
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
import sys, scipy
print(f'Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}')

# ── Project paths ────────────────────────────────────────────
# Default: the Colab Notebooks folder of Google Drive. Set THESIS_ROOT to run from any other folder.
ROOT     = Path(os.environ.get('THESIS_ROOT', '/content/drive/MyDrive/Colab Notebooks'))
if str(ROOT).startswith('/content/drive') and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
RAWDATA  = ROOT / 'Raw_Data'                  # Raw inputs
INTERMED = ROOT / 'thesis_processed'          # Intermediate computations
RESULTS  = ROOT / 'results'                   # Calibration and pricing results
FIGS     = ROOT / 'surface_plots'             # Per-snapshot surface plots

INTERMED.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print(f"Raw data folder       : {RAWDATA}")
print(f"Intermediate output   : {INTERMED}")
print(f"Results folder        : {RESULTS}")
print(f"Surface plots folder  : {FIGS}")
print()
print("=== Inventory of Raw_Data/ ===")
for f in sorted(RAWDATA.iterdir()):
    if f.is_file():
        mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name:<48} {mb:>8.1f} MB")


# Library versions used in this run (reported in Appendix A.2)
import sys, scipy, matplotlib
print('python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)


In [ ]:
# ============================================================
# Cell 1.2 - Define 74 snapshot dates
# Composition: 60 end-of-month + 14 crisis-event anchors
# Monthly cadence plus stress-event anchor dates
#              + crisis anchoring for regime variation
# ============================================================

# 60 end-of-month dates (2018-2022)
EOM = [
    "2018-01-31","2018-02-28","2018-03-29","2018-04-30",
    "2018-05-31","2018-06-29","2018-07-31","2018-08-31",
    "2018-09-28","2018-10-31","2018-11-30","2018-12-31",
    "2019-01-31","2019-02-28","2019-03-29","2019-04-30",
    "2019-05-31","2019-06-28","2019-07-31","2019-08-30",
    "2019-09-30","2019-10-31","2019-11-29","2019-12-31",
    "2020-01-31","2020-02-28","2020-03-31","2020-04-30",
    "2020-05-29","2020-06-30","2020-07-31","2020-08-31",
    "2020-09-30","2020-10-30","2020-11-30","2020-12-31",
    "2021-01-29","2021-02-26","2021-03-31","2021-04-30",
    "2021-05-28","2021-06-30","2021-07-30","2021-08-31",
    "2021-09-30","2021-10-29","2021-11-30","2021-12-31",
    "2022-01-31","2022-02-28","2022-03-31","2022-04-29",
    "2022-05-31","2022-06-30","2022-07-29","2022-08-31",
    "2022-09-30","2022-10-31","2022-11-30","2022-12-30",
]

# 15 crisis-event anchors (2020-02-28 coincides with a month-end date, so 60 + 14 = 74 dates)
EVENTS = {
    "2018-02-05": "Volmageddon: VIX ETN implosion (XIV/SVXY)",
    "2018-12-24": "Christmas Eve crash, SPX -20% YTD",
    "2019-08-05": "China tariff escalation, RMB devaluation",
    "2020-02-28": "COVID first wave, SPX -13% in one week",
    "2020-03-16": "VIX record close 82.69 (Cboe), fastest 30-day decline",
    "2020-03-23": "SPX trough at 2237, Fed unlimited QE announced",
    "2020-04-20": "WTI crude futures negative -305% intraday",
    "2021-01-27": "GameStop short squeeze peak, retail dealer stress",
    "2021-11-26": "Omicron variant Friday vol spike",
    "2022-01-24": "Fed hawkish pivot, intraday reversal +4%",
    "2022-02-24": "Russia invades Ukraine, energy crisis begins",
    "2022-06-13": "SPX in bear-market territory ahead of the 15 June 75bp FOMC hike",
    "2022-09-13": "August CPI shock, SPX -4.3% (largest since 2020)",
    "2022-09-26": "EUR/USD near its 20-year low, below parity (first broken in July 2022)",
    "2022-10-13": "September CPI surprise, intraday +5.5% reversal",
}

ALL_DATES = sorted(set(EOM) | set(EVENTS.keys()))
SNAP_SET  = set(ALL_DATES)

print(f"End-of-month dates : {len(EOM)}")
print(f"Crisis event dates : {len(EVENTS)}")
print(f"Combined unique    : {len(ALL_DATES)}")
print()
print("Crisis events:")
for d, label in sorted(EVENTS.items()):
    print(f"  {d}: {label}")


## Section 2. SPX options ingestion and Black-Scholes IV extraction

Three substeps:
1. Extract from optionsDX 7z archives, filter to 74 snapshot dates.
2. Convert wide to long format and apply standard liquidity filters (volume, minimum mid price, maturity and moneyness bounds).
3. Extract implied volatilities by inverting Black-Scholes via Brent root-finding (Hull 2022, §15.11). Cross-validate against the vendor's published IV (expected MAE < 1%, correlation > 0.99).

**Execution order:** 2.1 → 2.2 → 2.2b (rates) → 2.4 (dividend yield from put-call parity, needs r) → 2.5 (rates with the parity q) → 2.3 (implied volatility inversion, needs r and q).


In [ ]:
# ============================================================
# Cell 2.1 - SPX raw extraction from optionsDX 7z archives
# Output: SPX_options_clean.csv (filtered to 74 snapshot dates)
# ============================================================

EXTRACT_TMP = Path('/tmp/spx_extracted')
EXTRACT_TMP.mkdir(exist_ok=True)

def process_7z(path_7z):
    """Extract a 7z archive into temp, parse all txt/csv inside,
    keep only rows where the date column matches SNAP_SET."""
    extract_dir = EXTRACT_TMP / path_7z.stem
    extract_dir.mkdir(exist_ok=True)
    with py7zr.SevenZipFile(path_7z, mode='r') as z:
        z.extractall(path=extract_dir)
    frames = []
    for txt in sorted(extract_dir.glob('**/*.txt')) + sorted(extract_dir.glob('**/*.csv')):
        try:
            with open(txt, 'r') as f:
                head = f.readline()
            sep = ',' if head.count(',') > head.count('\t') else '\t'
            df = pd.read_csv(txt, sep=sep, low_memory=False, on_bad_lines='skip')
            df.columns = [c.strip().replace('[','').replace(']','').upper() for c in df.columns]
            date_col = next((c for c in df.columns
                             if c in ['QUOTE_DATE','DATE','QUOTEDATE','DATADATE']), None)
            if not date_col:
                continue
            df[date_col] = df[date_col].astype(str).str.strip()
            df = df[df[date_col].isin(SNAP_SET)]
            if not df.empty:
                frames.append(df)
        except Exception as e:
            print(f"    Error reading {txt.name}: {e}")
    shutil.rmtree(extract_dir)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print(f"Processing optionsDX archives for {len(ALL_DATES)} snapshot dates...")
spx_files = sorted(RAWDATA.glob('spx_eod_*.7z'))
print(f"Found {len(spx_files)} SPX archives in {RAWDATA.name}")

all_spx = []
for path_7z in spx_files:
    print(f"  [{path_7z.name}]", end=' ')
    raw_data = process_7z(path_7z)
    if not raw_data.empty:
        all_spx.append(raw_data)
        print(f"-> {len(raw_data):,} rows matched")
    else:
        print('-> no matches')

if not all_spx:
    raise RuntimeError("No SPX data extracted. Check Raw_Data/ folder and SNAP_SET dates.")

spx_clean = pd.concat(all_spx, ignore_index=True)
out_path = INTERMED / 'SPX_options_clean.csv'
spx_clean.to_csv(out_path, index=False)
print(f"\nSaved {len(spx_clean):,} rows to {out_path.name}")


In [ ]:
# ============================================================
# Cell 2.2 - Wide-to-long conversion + standard liquidity filters
# Standard liquidity filters: volume, minimum mid price, maturity and moneyness bounds
# Output: SPX_options_long.csv
# ============================================================

raw = pd.read_csv(INTERMED / 'SPX_options_clean.csv', low_memory=False)
print(f"Loaded {len(raw):,} rows | {len(raw.columns)} columns")

# Coerce numeric columns
NUM_COLS = ['UNDERLYING_LAST','STRIKE','C_BID','C_ASK','P_BID','P_ASK',
            'C_IV','P_IV','C_VOLUME','P_VOLUME',
            'C_DELTA','C_GAMMA','C_VEGA','C_THETA','C_RHO',
            'P_DELTA','P_GAMMA','P_VEGA','P_THETA','P_RHO']
for col in NUM_COLS:
    if col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')

date_col   = next((c for c in raw.columns
                    if 'QUOTE' in c.upper() and 'DATE' in c.upper()), None)
expiry_col = next((c for c in raw.columns
                    if 'EXPIRE' in c.upper() or 'EXPIR' in c.upper()), None)

keep_base = [c for c in [date_col, 'UNDERLYING_LAST', 'STRIKE', expiry_col] if c is not None]

# Calls
calls = raw[keep_base + ['C_BID','C_ASK','C_IV','C_VOLUME',
                          'C_DELTA','C_GAMMA','C_VEGA','C_THETA']].copy()
calls.columns = ['date','spot','strike','expiry',
                 'bid','ask','impl_vol','volume',
                 'delta','gamma','vega','theta']
calls['cp_flag'] = 'C'

# Puts
puts = raw[keep_base + ['P_BID','P_ASK','P_IV','P_VOLUME',
                         'P_DELTA','P_GAMMA','P_VEGA','P_THETA']].copy()
puts.columns = ['date','spot','strike','expiry',
                'bid','ask','impl_vol','volume',
                'delta','gamma','vega','theta']
puts['cp_flag'] = 'P'

spx = pd.concat([calls, puts], ignore_index=True)

# Derived fields
spx['date']      = pd.to_datetime(spx['date'], errors='coerce')
spx['expiry']    = pd.to_datetime(spx['expiry'], errors='coerce')
spx['ttm']       = (spx['expiry'] - spx['date']).dt.days / 365.0
spx['moneyness'] = pd.to_numeric(spx['strike'], errors='coerce') / pd.to_numeric(spx['spot'], errors='coerce')
spx['mid']       = (pd.to_numeric(spx['bid'], errors='coerce') + pd.to_numeric(spx['ask'], errors='coerce')) / 2.0
spx['volume']    = pd.to_numeric(spx['volume'], errors='coerce').fillna(0)
spx['impl_vol']  = pd.to_numeric(spx['impl_vol'], errors='coerce')
spx['spread']    = pd.to_numeric(spx['ask'], errors='coerce') - pd.to_numeric(spx['bid'], errors='coerce')

# OTM flag (calls if K > S, puts if K < S)
spx['is_otm']    = np.where(spx['cp_flag']=='C',
                             spx['strike'] > spx['spot'],
                             spx['strike'] < spx['spot'])

# Standard liquidity / arbitrage filters
n_before = len(spx)
spx = spx[
    (spx['volume']    >= 10)    &      # liquidity (liquidity filter)
    (spx['mid']       >= 0.05)  &      # price discreteness floor
    (spx['ttm']       >= 7/365) &      # exclude near-expiry liquidity bias
    (spx['ttm']       <= 2.0)   &      # standard 2Y horizon
    (spx['moneyness'] >= 0.70)  &      # remove far ITM/OTM with no volume
    (spx['moneyness'] <= 1.30)  &
    (spx['spread']    >= 0)     &      # remove inverted quotes
    (spx['is_otm']    == True)  &      # OTM only (more liquid, less American premium)
    (spx['spot']      >  0)     &
    spx['impl_vol'].notna()     &
    spx['date'].notna()         &
    spx['expiry'].notna()
].copy().reset_index(drop=True)

print(f"Wide-to-long total rows: {n_before:,}")
print(f"After OTM and filters  : {len(spx):,}")
print(f"Distinct dates         : {spx['date'].nunique()}")
print(f"Calls / Puts           : {(spx.cp_flag=='C').sum():,} / {(spx.cp_flag=='P').sum():,}")
print(f"Median IV              : {spx['impl_vol'].median()*100:.1f}%")

spx.to_csv(INTERMED / 'SPX_options_long.csv', index=False)
print(f"Saved: SPX_options_long.csv")


In [ ]:
# ============================================================
# Cell 2.2b - Supporting data and RATES_lookup (defines build_rates(); runs a first pass)
# Rates are built before the dividend yield and the implied volatilities:
#   build_rates() [r, r_f; q fallback] -> Cell 2.4 (q from put-call parity, needs r)
#   -> Cell 2.5: build_rates() again (q merged) -> Cell 2.3 (IV inversion needs r and q)
# Output: SPX_daily_prices, USD_rates, EUR_rates, RATES_lookup(.csv/.json)
# ============================================================
import yfinance as yf
import pandas_datareader.data as web
from dateutil.easter import easter


def target_business_days(start, end):
    """TARGET calendar: weekdays except 1 January, Good Friday, Easter Monday, 1 May, 25 and 26 December."""
    days = pd.date_range(start, end, freq='B')
    holidays = set()
    for y in range(days[0].year, days[-1].year + 1):
        e = pd.Timestamp(easter(y))
        holidays |= {pd.Timestamp(y, 1, 1), e - pd.Timedelta(days=2), e + pd.Timedelta(days=1),
                     pd.Timestamp(y, 5, 1), pd.Timestamp(y, 12, 25), pd.Timestamp(y, 12, 26)}
    return days[~days.isin(list(holidays))]


def euribor_as_of(monthly, days):
    """Monthly Euribor averages mapped to days with no look-ahead.

    A month's average is complete once its last TARGET fixing is published, so it is used from the
    last TARGET business day of that month onwards. Earlier in the month the latest complete average
    is the previous month's. monthly: Series indexed by monthly Period; days: DatetimeIndex.
    """
    tdays = target_business_days(days[0] - pd.offsets.MonthBegin(1), days[-1] + pd.offsets.MonthEnd(0))
    last_fixing = pd.Series(tdays, index=tdays).groupby(tdays.to_period('M')).max()
    periods = days.to_period('M')
    use = [p if d >= last_fixing[p] else p - 1 for d, p in zip(days, periods)]
    return pd.Series([monthly.get(p, np.nan) for p in use], index=days, dtype=float), pd.Index(use) != periods

def build_rates():
    global spx_raw, usd_rates, eur_rates, div_yield, lookup



    START, END = '2018-01-01', '2022-12-31'
    STUDY_START = pd.Timestamp(START)

    # 1. SPX daily close from FRED (SP500_dailyclose.csv in Raw_Data/)
    # Source: S&P Dow Jones Indices, redistributed via Federal Reserve FRED
    print("[1/5] SPX daily close from FRED (SP500_dailyclose.csv)...")
    spx_path = RAWDATA / 'SP500_dailyclose.csv'
    if not spx_path.exists():
        raise FileNotFoundError(
            f"{spx_path.name} not found in Raw_Data/. Download from "
            "https://fred.stlouisfed.org/series/SP500"
        )
    spx_raw = pd.read_csv(spx_path)
    # FRED CSV format: 'observation_date' / 'DATE' index column + 'SP500' value column
    date_col_name = next((c for c in spx_raw.columns
                           if c.lower() in ('date','observation_date','time_period')), spx_raw.columns[0])
    val_col_name  = next((c for c in spx_raw.columns
                           if c.upper() == 'SP500' or 'CLOSE' in c.upper()), spx_raw.columns[-1])
    spx_raw = spx_raw[[date_col_name, val_col_name]].rename(
        columns={date_col_name: 'Date', val_col_name: 'Close'}
    )
    spx_raw['Date']  = pd.to_datetime(spx_raw['Date'])
    spx_raw['Close'] = pd.to_numeric(spx_raw['Close'], errors='coerce')
    spx_raw = spx_raw.dropna(subset=['Close']).set_index('Date').sort_index()
    spx_raw['LogReturn'] = np.log(spx_raw['Close'] / spx_raw['Close'].shift(1))
    spx_raw.to_csv(INTERMED / 'SPX_daily_prices.csv')
    print(f"  {len(spx_raw):,} trading days")
    print(f"  Range : [{spx_raw['Close'].min():.0f}, {spx_raw['Close'].max():.0f}]")
    print(f"  Window: {spx_raw.index[0].date()} to {spx_raw.index[-1].date()}")

    # 2. USD risk-free rates from FRED
    print("\n[2/5] USD rates from FRED (DGS3MO, DGS2, DGS10)...")
    usd_rates = pd.DataFrame()
    for series, name in [('DGS3MO','USD_3M'),('DGS2','USD_2Y'),('DGS10','USD_10Y')]:
        try:
            s = web.DataReader(series, 'fred', START, END)
            usd_rates[name] = s.squeeze() / 100.0
            print(f"  {series}: {s.shape[0]:,} rows")
        except Exception as e:
            print(f"  WARNING: {series} failed: {e}")

    usd_rates = usd_rates.ffill().bfill()
    usd_rates.to_csv(INTERMED / 'USD_rates.csv')

    # 3. EUR Euribor 3M from local ECB Data Portal CSV (Raw_Data/ECB_EURIBOR_Data.csv)
    # Series: FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA (monthly historical close, period average)
    # Source: ECB Data Portal (Euribor, EMMI benchmark)
    # Each business day receives the latest monthly average complete on that day (euribor_as_of).
    print("\n[3/5] EUR Euribor 3M from local ECB CSV (FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA)...")
    ecb_path = RAWDATA / 'ECB_EURIBOR_Data.csv'
    if not ecb_path.exists():
        raise FileNotFoundError(
            f"{ecb_path.name} not found in Raw_Data/. "
            "Download from https://data.ecb.europa.eu/data/datasets/FM/"
            "FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA"
        )
    ecb_raw = pd.read_csv(ecb_path)
    print(f"  Raw file rows : {len(ecb_raw)} (1994 to 2026 history)")

    # ECB Data Portal CSV columns are typically: 'DATE','TIME PERIOD','<series description>'
    # The value column has a long descriptive header; locate it positionally
    date_col = next((c for c in ecb_raw.columns if c.upper() == 'DATE'), None)
    val_col  = ecb_raw.columns[-1]   # last column is always the value
    if date_col is None:
        date_col = ecb_raw.columns[0]
    ecb = pd.DataFrame({
        'date': pd.to_datetime(ecb_raw[date_col], errors='coerce'),
        'Euribor_3M_pct': pd.to_numeric(ecb_raw[val_col], errors='coerce'),
    }).dropna().set_index('date').sort_index()
    # Convert percent to decimal
    ecb['Euribor_3M'] = ecb['Euribor_3M_pct'] / 100.0

    # Slice to study window
    ecb_window = ecb.loc[START:END, ['Euribor_3M']].copy()
    print(f"  In study window 2018-2022: {len(ecb_window)} monthly observations")
    print(f"  Range : [{ecb_window['Euribor_3M'].min()*100:.3f}%, "
          f"{ecb_window['Euribor_3M'].max()*100:.3f}%]")
    print(f"  Mean  : {ecb_window['Euribor_3M'].mean()*100:.3f}%")

    # Forward-fill from monthly to daily (business days)
    trading_days_eur = pd.date_range(START, END, freq='B')
    eur_rates = pd.DataFrame(index=trading_days_eur)
    # The ECB file stamps each monthly average at the month's last day. The full history is passed
    # so that the first days of the window can fall back on December 2017.
    month_avg = ecb['Euribor_3M'].copy()
    month_avg.index = month_avg.index.to_period('M')
    month_avg = month_avg.groupby(level=0).last()
    eur_rates['Euribor_3M'], on_prev = euribor_as_of(month_avg, trading_days_eur)
    eur_rates = eur_rates.ffill().bfill()
    eur_rates.to_csv(INTERMED / 'EUR_rates.csv')
    print(f"  Mapped to {len(eur_rates):,} business days "
          f"({on_prev.mean():.0%} on the previous month's average, the rest on a complete own-month average)")

    # Sanity check vs published values
    print(f"\n  Sanity check (key dates):")
    for ck_date in ['2018-01-31', '2020-03-16', '2020-03-31', '2022-09-26', '2022-12-30']:
        d = pd.Timestamp(ck_date)
        if d in eur_rates.index:
            v = eur_rates.loc[d, 'Euribor_3M'] * 100
            print(f"    {ck_date}: {v:>7.3f}%")

    # 4. SPX dividend yield from put-call parity (Hull 2022, §17.4, eq. 17.10)
    # Computed in Cell 2.4 from the option chain itself - forward-looking,
    # internally consistent with the IV surface
    print("\n[4/5] SPX dividend yield from put-call parity (Hull 2022, §17.4)...")
    imp_div_path = INTERMED / 'SPX_implied_dividend_yield.csv'
    if imp_div_path.exists():
        div_snap = pd.read_csv(imp_div_path, index_col=0, parse_dates=True)
        # Reindex onto trading_days, forward-fill between snapshots
        div_yield = pd.DataFrame(index=pd.date_range(START, END, freq='B'))
        div_yield['q'] = div_snap['q'].reindex(div_yield.index, method='ffill').bfill()
        print(f"  Implied q (parity)    : {len(div_snap)} snapshot values")
        print(f"  Forward-filled to     : {len(div_yield)} trading days")
        print(f"  q range               : [{div_yield['q'].min()*100:.2f}%, {div_yield['q'].max()*100:.2f}%]")
        print(f"  q mean                : {div_yield['q'].mean()*100:.2f}%")
    else:
        print("  WARNING: SPX_implied_dividend_yield.csv not found")
        print("  Run Cell 2.4 first to extract dividends via put-call parity.")
        print("  Falling back to 1.8% period-average constant.")
        idx = pd.date_range(START, END, freq='B')
        div_yield = pd.DataFrame({'q': 0.018}, index=idx)

    # 5. Build per-day RATES_lookup
    print("\n[5/5] Building RATES_lookup per trading day...")
    trading_days = pd.date_range(STUDY_START, END, freq='B')
    lookup = pd.DataFrame(index=trading_days)
    lookup.index.name = 'date'

    if 'USD_3M' in usd_rates.columns:
        lookup['r'] = usd_rates['USD_3M'].reindex(trading_days, method='ffill')
    else:
        lookup['r'] = 0.025
    lookup['q'] = div_yield['q'].reindex(trading_days, method='ffill')
    if 'Euribor_3M' in eur_rates.columns:
        lookup['r_f'] = eur_rates['Euribor_3M'].reindex(trading_days, method='ffill')
    else:
        lookup['r_f'] = -0.005

    # Add full USD curve for downstream uses
    for col in ['USD_3M','USD_2Y','USD_10Y']:
        if col in usd_rates.columns:
            lookup[col] = usd_rates[col].reindex(trading_days, method='ffill')

    lookup = lookup.ffill().bfill()
    lookup.to_csv(INTERMED / 'RATES_lookup.csv')

    # Save JSON for fast date lookup in Cell 2.3
    lookup_dict = {
        dt.strftime('%Y-%m-%d'): {
            'r':   round(float(row['r']),   6),
            'q':   round(float(row['q']),   6),
            'r_f': round(float(row['r_f']), 6),
        }
        for dt, row in lookup.iterrows()
    }
    with open(INTERMED / 'RATES_lookup.json', 'w') as f:
        json.dump(lookup_dict, f)

    print(f"\nRATES_lookup table:")
    print(f"  Trading days : {len(lookup):,}")
    print(f"  r (USD 3M)   : [{lookup['r'].min()*100:.2f}%, {lookup['r'].max()*100:.2f}%]")
    print(f"  q (SPX div)  : [{lookup['q'].min()*100:.2f}%, {lookup['q'].max()*100:.2f}%]")
    print(f"  r_f (Euribor): [{lookup['r_f'].min()*100:.2f}%, {lookup['r_f'].max()*100:.2f}%]")
    print(f"\nSaved: RATES_lookup.csv + RATES_lookup.json")


build_rates()

### Implied dividend yield from put-call parity

For European index options, put-call parity gives:

$$C(K,T) - P(K,T) = S \cdot e^{-qT} - K \cdot e^{-rT}$$

Solving for the continuous dividend yield q:

$$q = -\frac{1}{T} \ln\!\left( \frac{C - P + K \cdot e^{-rT}}{S} \right)$$

This produces a forward-looking dividend yield estimate that is **internally consistent with the option chain itself**, following the put-call parity relation for European index options (Hull 2022, §17.4). The parity yield is the carry implied by the option market (it also absorbs any gap between the T-bill rate and the financing rate embedded in SPX options); 27 of 72 values fall outside 1-3%. For each snapshot date and maturity bucket we extract q from near-ATM strikes (where put-call parity is most reliable) and take the median to mitigate bid-ask noise. The output replaces the placeholder `q` column in `RATES_lookup` so that downstream pricing uses dividend yields consistent with the same option chains they're calibrated to.


In [ ]:
# ============================================================
# Cell 2.4 - Implied dividend yield via put-call parity (Hull 2022, §17.4)
#
# Reads SPX_options_clean.csv (WIDE format from Cell 2.1) where each row
# already contains both call (C_BID/C_ASK) and put (P_BID/P_ASK) columns
# at the same strike on the same date - no merge required.
# ============================================================

# Load wide-format file (pre-OTM-filter, calls and puts on same row)
raw_wide = pd.read_csv(INTERMED / 'SPX_options_clean.csv', low_memory=False)
print(f"Loaded wide format: {len(raw_wide):,} rows | {len(raw_wide.columns)} columns")

# Coerce numeric columns we will use
for col in ['UNDERLYING_LAST', 'STRIKE',
            'C_BID', 'C_ASK', 'P_BID', 'P_ASK',
            'C_VOLUME', 'P_VOLUME']:
    if col in raw_wide.columns:
        raw_wide[col] = pd.to_numeric(raw_wide[col], errors='coerce')

# Identify date and expiry columns
date_col   = next((c for c in raw_wide.columns
                    if 'QUOTE' in c.upper() and 'DATE' in c.upper()), None)
expiry_col = next((c for c in raw_wide.columns
                    if 'EXPIRE' in c.upper() or 'EXPIR' in c.upper()), None)

# Build a clean parity dataframe with both legs at each (date, expiry, strike)
# No merge needed because the wide format already pairs C and P at the same strike
parity = pd.DataFrame({
    'date':   pd.to_datetime(raw_wide[date_col],   errors='coerce'),
    'expiry': pd.to_datetime(raw_wide[expiry_col], errors='coerce'),
    'strike': raw_wide['STRIKE'],
    'spot':   raw_wide['UNDERLYING_LAST'],
    'C_mid': (raw_wide['C_BID'] + raw_wide['C_ASK']) / 2.0,
    'P_mid': (raw_wide['P_BID'] + raw_wide['P_ASK']) / 2.0,
    'C_vol': raw_wide.get('C_VOLUME', pd.Series([0]*len(raw_wide))).fillna(0),
    'P_vol': raw_wide.get('P_VOLUME', pd.Series([0]*len(raw_wide))).fillna(0),
})
parity['ttm']       = (parity['expiry'] - parity['date']).dt.days / 365.0
parity['moneyness'] = parity['strike'] / parity['spot']

# Initial sanity: drop rows missing either leg or with zero/NaN spot/strike
n0 = len(parity)
parity = parity.dropna(subset=['date','expiry','strike','spot','C_mid','P_mid','ttm']).copy()
parity = parity[(parity['spot'] > 0) & (parity['strike'] > 0)
                & (parity['C_mid'] > 0) & (parity['P_mid'] > 0)
                & (parity['ttm'] > 0)].copy()
print(f"After dropna and positive-price filter: {len(parity):,} rows (from {n0:,})")

# Get USD 3M rate per date from RATES_lookup if available
try:
    rates_existing = pd.read_csv(INTERMED / 'RATES_lookup.csv',
                                   index_col=0, parse_dates=True).ffill()
    use_lookup = True
    print("Using r from existing RATES_lookup.csv")
except FileNotFoundError:
    use_lookup = False
    print("RATES_lookup.csv not found - using r=2.5% constant fallback")

def get_r(d):
    if not use_lookup:
        return 0.025
    rows = rates_existing.loc[rates_existing.index <= d]
    if rows.empty:
        return 0.025
    r = float(rows['r'].iloc[-1])
    return r if np.isfinite(r) else 0.025

# Apply r per row
parity['r'] = parity['date'].apply(get_r)

# Put-call parity inversion (Hull 2022, §17.4):
#   C - P = S * exp(-q*T) - K * exp(-r*T)
# Solving for q:
#   q = -(1/T) * ln[(C - P + K*exp(-r*T)) / S]
inner = (parity['C_mid'] - parity['P_mid']
         + parity['strike'] * np.exp(-parity['r'] * parity['ttm'])) / parity['spot']
# Honest handling: drop rows where parity inversion is undefined
parity['q_implied'] = np.where(inner > 0,
                                 -np.log(inner.where(inner > 0, np.nan)) / parity['ttm'],
                                 np.nan)

# Filter to where parity is most reliable:
#   - Near-ATM strikes (most liquid, tightest spreads)
#   - Maturities between 30 days and 1.5 years
#   - Both legs traded in the day (volume > 0 on each)
#   - Plausible q range to drop calculation outliers
clean = parity[
    (parity['moneyness'] >= 0.95) & (parity['moneyness'] <= 1.05) &
    (parity['ttm'] >= 30/365) & (parity['ttm'] <= 1.5) &
    (parity['C_vol'] > 0) & (parity['P_vol'] > 0) &
    parity['q_implied'].between(-0.01, 0.06) &
    parity['q_implied'].notna() & np.isfinite(parity['q_implied'])
].copy()
print(f"After moneyness, ttm, volume and plausibility filters: {len(clean):,} rows")
print()

# ── Two-stage aggregation: median across strikes per expiry, then across expiries ──
# Stage 1: median q per (date, expiry) - parity within a single maturity
# Stage 2: median across expiries per date - dampens single-maturity noise
print("Two-stage aggregation:")
clean['date_str']   = clean['date'].dt.strftime('%Y-%m-%d')
clean['expiry_str'] = clean['expiry'].dt.strftime('%Y-%m-%d')

stage1 = (clean.groupby(['date_str','expiry_str'])
                .agg(q_expiry=('q_implied', 'median'),
                     n_strikes=('q_implied', 'count'))
                .reset_index())
stage1 = stage1[stage1['n_strikes'] >= 3].copy()
print(f"  Stage 1: {len(stage1):,} (date, expiry) groups with >=3 ATM strikes")

q_by_date = (stage1.groupby('date_str')
                    .agg(q=('q_expiry', 'median'),
                         n_expiries=('q_expiry', 'count'))
                    .reset_index())
q_by_date['date'] = pd.to_datetime(q_by_date['date_str'])
q_by_date = q_by_date.set_index('date')[['q','n_expiries']]
print(f"  Stage 2: {len(q_by_date)} snapshot dates with implied q")
print()

print(f"Implied dividend yield by snapshot date:")
print(f"  Snapshots with q   : {len(q_by_date)}")
print(f"  q mean             : {q_by_date['q'].mean()*100:.2f}%")
print(f"  q median           : {q_by_date['q'].median()*100:.2f}%")
print(f"  q range            : [{q_by_date['q'].min()*100:.2f}%, {q_by_date['q'].max()*100:.2f}%]")
print(f"  Mean expiries used : {q_by_date['n_expiries'].mean():.1f} per date")

# Sanity: 2018-2022 SPX yield was approximately 1.4% to 2.3% per S&P-published data
expected_low, expected_high = 0.010, 0.030
n_sane = ((q_by_date['q'] >= expected_low) & (q_by_date['q'] <= expected_high)).sum()
print(f"\nSanity (q in [1.0%, 3.0%]): {n_sane}/{len(q_by_date)} dates pass")
if n_sane < 0.7 * len(q_by_date):
    print("  WARNING: many dates outside expected range. Inspect q_by_date manually.")

# Save (only the 'q' column, n_expiries kept for diagnostics in a separate file)
q_by_date[['q']].to_csv(INTERMED / 'SPX_implied_dividend_yield.csv')
q_by_date.to_csv(INTERMED / 'SPX_implied_dividend_yield_diagnostics.csv')
print(f"\nSaved: SPX_implied_dividend_yield.csv (q column only, used by Cell 5.1)")
print(f"Saved: SPX_implied_dividend_yield_diagnostics.csv (with n_expiries audit column)")
print()
print("Run Cell 2.5 next so that RATES_lookup picks up the implied q values.")


In [ ]:
# ============================================================
# Cell 2.5 - Rebuild RATES_lookup with the parity dividend yield (second pass of build_rates)
# ============================================================
build_rates()


In [ ]:
# ============================================================
# Cell 2.3 - Black-Scholes IV extraction via Brent root-finding
# Hull (2022), §15.11; uses date-specific r, q, r_f from RATES_lookup
# (run after Cell 2.5: needs r, r_f and the parity q in RATES_lookup.json)
# Output: SPX_options_iv.csv
# ============================================================

spx = pd.read_csv(INTERMED / 'SPX_options_long.csv', parse_dates=['date','expiry'])
spx['ttm']       = (spx['expiry'] - spx['date']).dt.days / 365.0
spx['moneyness'] = spx['strike'] / spx['spot']
spx['mid']       = (spx['bid'] + spx['ask']) / 2.0

try:
    with open(INTERMED / 'RATES_lookup.json') as f:
        rates_lut = json.load(f)
    USE_RATE_LUT = True
    print("Date-specific rates loaded from RATES_lookup.json")
except FileNotFoundError:
    rates_lut = {}
    USE_RATE_LUT = False
    print("RATES_lookup.json not found - using period-average constants")
    print("(run Cell 5.1 first to enable date-specific rates)")

def get_rates_for_date(date_str):
    if USE_RATE_LUT and date_str in rates_lut:
        d = rates_lut[date_str]
        return d['r'], d['q'], d['r_f']
    return 0.025, 0.018, -0.002   # 2018-2022 period averages as fallback

def bs_price(S, K, T, r, sigma, q, flag):
    """Black-Scholes-Merton price with continuous dividend yield."""
    if T <= 0 or sigma <= 0:
        return 0.0
    d1 = (math.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    if flag == 'C':
        return S*math.exp(-q*T)*norm.cdf(d1) - K*math.exp(-r*T)*norm.cdf(d2)
    return K*math.exp(-r*T)*norm.cdf(-d2) - S*math.exp(-q*T)*norm.cdf(-d1)

def extract_iv(mid, S, K, T, r, q, flag):
    """Invert Black-Scholes via Brent root-finding."""
    if T <= 0 or mid <= 0 or S <= 0 or K <= 0:
        return np.nan
    try:
        f = lambda s: bs_price(S, K, T, r, s, q, flag) - mid
        if f(1e-4) * f(5.0) > 0:
            return np.nan
        return brentq(f, 1e-4, 5.0, xtol=1e-8)
    except Exception:
        return np.nan

print(f"Extracting IV for {len(spx):,} contracts...")
ivs = []
for _, row in spx.iterrows():
    d_str = row['date'].strftime('%Y-%m-%d')
    r, q, r_f = get_rates_for_date(d_str)
    iv = extract_iv(float(row['mid']), float(row['spot']), float(row['strike']),
                    float(row['ttm']), r, q, str(row['cp_flag']))
    ivs.append(iv)

spx['iv_extracted'] = ivs

# Cross-validation against vendor IV
valid = spx['iv_extracted'].notna() & spx['impl_vol'].notna()
mae   = (spx.loc[valid,'iv_extracted'] - spx.loc[valid,'impl_vol']).abs().mean()
corr  = spx.loc[valid,['iv_extracted','impl_vol']].corr().iloc[0,1]

print(f"\nSuccessful extractions    : {spx['iv_extracted'].notna().sum():,} / {len(spx):,}")
print(f"Cross-check vs vendor IV  :")
print(f"  MAE      = {mae*100:.4f}% volatility points")
print(f"  Correlation = {corr:.6f}")
print(f"  Expected MAE < 1%, correlation > 0.99")

# IV used downstream: prefer extracted, fallback to vendor
spx['iv_thesis'] = spx['iv_extracted'].fillna(spx['impl_vol'])
spx.to_csv(INTERMED / 'SPX_options_iv.csv', index=False)
print(f"\nSaved: SPX_options_iv.csv ({len(spx):,} rows)")


## Section 3. SPX volatility surface construction

Linear interpolation on a Delaunay triangulation (scipy `griddata`) onto a 17 × 9 moneyness × time-to-maturity grid. Quotes beyond 2 years are filtered and griddata does not extrapolate, so the last populated SPX pillar is 504 days (1.38y). Smoothing (SVI) and the arbitrage checks are done later, in Cell 9.1. For sparse dates where the post-filter point cloud doesn't fully span the grid, the cell falls back to nearest-neighbour interpolation to fill empty cells.


In [ ]:
# ============================================================
# Cell 3.1 - SPX volatility surface (linear on Delaunay triangulation + NN fallback)
# Output: SPX_vol_surface_grid.csv + 3D plots in FIGS folder
# ============================================================

spx = pd.read_csv(INTERMED / 'SPX_options_iv.csv', parse_dates=['date','expiry'])

MONEYNESS_GRID = np.array([0.70,0.75,0.80,0.85,0.90,0.925,0.95,0.975,
                            1.00,1.025,1.05,1.075,1.10,1.15,1.20,1.25,1.30])
TTM_GRID_DAYS  = np.array([30,60,90,120,180,252,365,504,730])
TTM_GRID       = TTM_GRID_DAYS / 365.0

CRISIS_DATES = {'2020-03-16','2020-03-23','2022-09-26'}

surf_records = []
dates_with_surface = []
sparse_dates = []

snap_dates = sorted(spx['date'].dt.strftime('%Y-%m-%d').unique())
print(f"Building surfaces for {len(snap_dates)} snapshot dates...")

for d in snap_dates:
    sub = spx[spx['date'].dt.strftime('%Y-%m-%d') == d].dropna(subset=['iv_thesis'])
    if len(sub) < 10:
        print(f"  SKIP {d}: only {len(sub)} contracts after filtering")
        continue

    pts = sub[['moneyness','ttm']].values
    ivs = sub['iv_thesis'].values
    gK, gT = np.meshgrid(MONEYNESS_GRID, TTM_GRID)
    try:
        grid_iv = griddata(pts, ivs, (gK, gT), method='linear')
    except Exception:
        grid_iv = np.full(gK.shape, np.nan)

    # Nearest-neighbor fallback for sparse cells
    nan_frac = np.isnan(grid_iv).mean()
    if nan_frac > 0.5:
        sparse_dates.append((d, nan_frac))
        try:
            grid_iv_nn = griddata(pts, ivs, (gK, gT), method='nearest')
            mask = np.isnan(grid_iv)
            grid_iv[mask] = grid_iv_nn[mask]
        except Exception:
            continue

    if np.all(np.isnan(grid_iv)):
        continue

    for i, ttm in enumerate(TTM_GRID):
        for j, k in enumerate(MONEYNESS_GRID):
            v = grid_iv[i,j]
            if not np.isnan(v):
                surf_records.append({'date':d, 'ttm_years':ttm, 'moneyness':k, 'impl_vol':v})

    # 3D plot
    is_crisis = d in CRISIS_DATES
    fig = plt.figure(figsize=(12,6))
    ax  = fig.add_subplot(111, projection='3d')
    cmap = plt.cm.Reds if is_crisis else plt.cm.Blues_r
    ax.plot_surface(gK*100, gT*365, grid_iv*100, cmap=cmap, alpha=0.85, edgecolor='none')
    ax.set_xlabel('Moneyness K/S (%)'); ax.set_ylabel('TTM (days)'); ax.set_zlabel('IV (%)')
    title = f"SPX Implied Volatility Surface -- {d}"
    if is_crisis: title += ' (crisis)'
    ax.set_title(title, fontsize=11)
    plt.savefig(FIGS / f"spx_vol_surface_{d.replace('-','')}.png",
                dpi=120, bbox_inches='tight')
    plt.close()
    dates_with_surface.append(d)

surface_df = pd.DataFrame(surf_records)
surface_df.to_csv(INTERMED / 'SPX_vol_surface_grid.csv', index=False)
print(f"\nSurfaces built       : {len(dates_with_surface)} dates")
print(f"Grid records (rows)  : {len(surface_df):,}")
if sparse_dates:
    print(f"NN fallback applied  : {len(sparse_dates)} dates")
    for d, frac in sparse_dates:
        print(f"  {d}: {frac*100:.0f}% of grid filled by nearest-neighbour")
print(f"3D plots saved       : {FIGS.name}/spx_vol_surface_*.png")


## Section 4. VIX extraction from optionsDX archives

Cell 4.1 extracts the optionsDX VIX level (quote-time value). Cell 4.2 replaces it with the official Cboe daily close, which is what the regime classification uses. The optionsDX vix_eod archives contain VIX option chains plus the underlying VIX level itself; here we extract the daily VIX level series.


In [ ]:
# ============================================================
# Cell 4.1 - VIX daily level extraction
# Output: VIX_daily.csv with columns Date, Close
# ============================================================

VIX_TMP = Path('/tmp/vix_extracted')
VIX_TMP.mkdir(exist_ok=True)

def extract_vix_archive(path_7z):
    extract_dir = VIX_TMP / path_7z.stem
    extract_dir.mkdir(exist_ok=True)
    with py7zr.SevenZipFile(path_7z, mode='r') as z:
        z.extractall(path=extract_dir)
    frames = []
    for txt in sorted(extract_dir.glob('**/*.txt')) + sorted(extract_dir.glob('**/*.csv')):
        try:
            with open(txt, 'r') as f:
                head = f.readline()
            sep = ',' if head.count(',') > head.count('\t') else '\t'
            df = pd.read_csv(txt, sep=sep, low_memory=False, on_bad_lines='skip')
            df.columns = [c.strip().replace('[','').replace(']','').upper() for c in df.columns]
            # The optionsDX VIX file has UNDERLYING_LAST as the VIX level
            # plus QUOTE_DATE; we collapse to one row per date
            date_col = next((c for c in df.columns if 'QUOTE' in c and 'DATE' in c), None)
            level_col = next((c for c in df.columns if c == 'UNDERLYING_LAST'), None)
            if date_col and level_col:
                vix_levels = df[[date_col, level_col]].drop_duplicates(subset=[date_col])
                vix_levels.columns = ['Date', 'Close']
                vix_levels['Close'] = pd.to_numeric(vix_levels['Close'], errors='coerce')
                frames.append(vix_levels)
        except Exception as e:
            print(f"    Error: {e}")
    shutil.rmtree(extract_dir)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

vix_files = sorted(RAWDATA.glob('vix_eod_*.7z'))
print(f"Found {len(vix_files)} VIX archives in {RAWDATA.name}")

vix_all = []
for f7z in vix_files:
    print(f"  [{f7z.name}]", end=' ')
    df_vix = extract_vix_archive(f7z)
    if not df_vix.empty:
        vix_all.append(df_vix)
        print(f"-> {len(df_vix):,} unique dates")
    else:
        print('-> empty')

if vix_all:
    vix_full = pd.concat(vix_all, ignore_index=True).drop_duplicates(subset=['Date'])
    vix_full['Date'] = pd.to_datetime(vix_full['Date'])
    vix_full = vix_full.sort_values('Date').set_index('Date')
    vix_full.to_csv(INTERMED / 'VIX_daily.csv')
    print(f"\nVIX series         : {len(vix_full):,} trading days")
    print(f"Min / Max          : {vix_full['Close'].min():.2f} / {vix_full['Close'].max():.2f}")
    print(f"Date range         : {vix_full.index[0].date()} to {vix_full.index[-1].date()}")

    # Sanity check: VIX peak in March 2020
    march_2020 = vix_full[(vix_full.index >= '2020-03-01') & (vix_full.index <= '2020-03-31')]
    if len(march_2020) > 0:
        peak_date = march_2020['Close'].idxmax()
        peak_val  = march_2020['Close'].max()
        print(f"March 2020 peak    : {peak_val:.2f} on {peak_date.date()}")
        print(f"  (literature reports 82.69 intraday on 2020-03-16)")
    print(f"Saved              : VIX_daily.csv")
else:
    print("\nNo VIX data extracted. Falling back to yfinance...")
    import yfinance as yf
    vix = yf.download('^VIX', start='2018-01-01', end='2022-12-31',
                       progress=False, auto_adjust=False)
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = vix.columns.get_level_values(0)
    vix.to_csv(INTERMED / 'VIX_daily.csv')
    print(f"  Pulled from yfinance: {len(vix):,} rows")


In [ ]:
# ============================================================
# Cell 4.2 - Official Cboe VIX daily close (overrides Cell 4.1 for regime classification)
# Source: Cboe Global Markets, VIX historical data,
#         https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv
# Why: the optionsDX archive reports VIX at the option quote time, not the official close
#      (e.g. 2019-06-28: optionsDX 15.60 vs Cboe close 15.08). Several snapshots sit within
#      0.5 points of the 20/30 thresholds, so the official close is used for classification.
# Output: VIX_daily.csv (Date, Close) - Cboe close; VIX_optionsDX.csv keeps the Cell 4.1 series
# ============================================================
CBOE_URL = 'https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv'
old_vix_path = INTERMED / 'VIX_daily.csv'
if old_vix_path.exists() and not (INTERMED / 'VIX_optionsDX.csv').exists():
    shutil.copy(old_vix_path, INTERMED / 'VIX_optionsDX.csv')
try:
    cboe = pd.read_csv(CBOE_URL)
    print('Downloaded Cboe VIX history:', len(cboe), 'rows')
except Exception as e:
    local = RAWDATA / 'VIX_History.csv'
    print(f'Download failed ({e}); reading {local.name} from Raw_Data/ (save the Cboe CSV there)')
    cboe = pd.read_csv(local)
cboe.columns = [c.strip().upper() for c in cboe.columns]
cboe['DATE'] = pd.to_datetime(cboe['DATE'])
vix_cboe = (cboe.set_index('DATE')['CLOSE'].astype(float).loc['2017-12-01':'2023-01-31']
            .rename('Close').to_frame())
vix_cboe.index.name = 'Date'
vix_cboe.to_csv(INTERMED / 'VIX_daily.csv')
print(f"Cboe VIX close 2020-03-16 = {vix_cboe.loc['2020-03-16','Close']:.2f} (record close)")
if (INTERMED / 'VIX_optionsDX.csv').exists():
    vo = pd.read_csv(INTERMED / 'VIX_optionsDX.csv', index_col=0, parse_dates=True).iloc[:, 0]
    snaps = [pd.Timestamp(d) for d in ALL_DATES]
    cmp_ = pd.DataFrame({'optionsDX': vo.reindex(snaps), 'Cboe_close': vix_cboe['Close'].reindex(snaps)})
    cmp_['diff'] = cmp_.optionsDX - cmp_.Cboe_close
    print('Snapshot dates, optionsDX minus Cboe close: mean %.2f, max abs %.2f' % (cmp_['diff'].mean(), cmp_['diff'].abs().max()))


## Section 5. Public market data

USD Treasury rates (FRED), 3-month Euribor (ECB) and the S&P 500 daily close (FRED) are loaded by `build_rates()` in Cell 2.2b, because the rates are needed before the dividend yield and the implied volatilities are computed.


## Section 6. EUR/USD spot from LSEG and paired log-returns

Reads the LSEG-sourced EUR/USD spot file from Raw_Data, computes mid from BID/ASK, and builds paired log-returns (SPX from FRED SP500 + EUR/USD from LSEG) for copula calibration.

This section ensures methodological consistency on the FX side: both EUR/USD spot and EUR/USD vol composite come from LSEG REFINITIV, eliminating any cross-vendor consistency concern in the Garman-Kohlhagen forward calculation.


In [ ]:
# ============================================================
# Cell 6.1 - EUR/USD spot from LSEG, paired log-returns
# Input: Raw_Data/EURUSD_spot_LSEG.csv
# Output: EURUSD_daily_spot.csv, PAIRED_returns_copula.csv
# ============================================================

LSEG_SPOT = RAWDATA / 'EURUSD_spot_LSEG.csv'
if not LSEG_SPOT.exists():
    raise FileNotFoundError(f"{LSEG_SPOT.name} not found in Raw_Data/")
print(f"Loading LSEG spot file: {LSEG_SPOT.name}")

# LSEG file has multi-row header [RIC, Field]
df_spot_raw = pd.read_csv(LSEG_SPOT, header=[0, 1], index_col=0, parse_dates=True)
bid_col = [c for c in df_spot_raw.columns if c[1].upper() == 'BID']
ask_col = [c for c in df_spot_raw.columns if c[1].upper() == 'ASK']
if not bid_col or not ask_col:
    df_spot_raw = pd.read_csv(LSEG_SPOT, index_col=0, parse_dates=True)
    eurusd_spot = df_spot_raw.iloc[:, 0].astype(float)
else:
    eurusd_spot = (df_spot_raw[bid_col[0]] + df_spot_raw[ask_col[0]]) / 2.0

eurusd = pd.DataFrame({'EURUSD': eurusd_spot.dropna().astype(float)})
eurusd['LogReturn']    = np.log(eurusd['EURUSD'] / eurusd['EURUSD'].shift(1))
eurusd['RealVol_21d']  = eurusd['LogReturn'].rolling(21).std()  * np.sqrt(252)
eurusd['RealVol_63d']  = eurusd['LogReturn'].rolling(63).std()  * np.sqrt(252)
eurusd['RealVol_252d'] = eurusd['LogReturn'].rolling(252).std() * np.sqrt(252)
eurusd.to_csv(INTERMED / 'EURUSD_daily_spot.csv')
print(f"  EUR/USD daily spot   : {len(eurusd):,} rows")
print(f"  Range                : [{eurusd['EURUSD'].min():.4f}, {eurusd['EURUSD'].max():.4f}]")

# Paired log-returns
spx_daily = pd.read_csv(INTERMED / 'SPX_daily_prices.csv', index_col=0, parse_dates=True)
if isinstance(spx_daily.columns, pd.MultiIndex):
    spx_daily.columns = spx_daily.columns.get_level_values(0)

spx_ret = spx_daily['LogReturn'].dropna()
spx_ret.name = 'SPX_LogReturn'
eur_ret = eurusd['LogReturn'].dropna()
eur_ret.name = 'EUR_LogReturn'

paired = pd.concat([spx_ret, eur_ret], axis=1).dropna()
paired.columns = ['SPX_LogReturn', 'EUR_LogReturn']
paired['Roll_Corr_60d']  = paired['SPX_LogReturn'].rolling(60).corr(paired['EUR_LogReturn'])
paired['Roll_Corr_252d'] = paired['SPX_LogReturn'].rolling(252).corr(paired['EUR_LogReturn'])
paired.to_csv(INTERMED / 'PAIRED_returns_copula.csv')

# Diagnostics
rho = paired[['SPX_LogReturn','EUR_LogReturn']].corr().iloc[0,1]
tau = stats.kendalltau(paired['SPX_LogReturn'], paired['EUR_LogReturn'])[0]
spr = stats.spearmanr(paired['SPX_LogReturn'], paired['EUR_LogReturn'])[0]
print(f"\nPaired observations  : {len(paired):,}")
print(f"Pearson rho          : {rho:.4f}  (linear, benchmark)")
print(f"Kendall tau          : {tau:.4f}  (Nelsen 2006, Thm 5.1.3)")
print(f"Spearman rho         : {spr:.4f}")
print(f"\nSaved: EURUSD_daily_spot.csv, PAIRED_returns_copula.csv")


## Section 7. EUR/USD volatility surface from LSEG OTC composite

EUR/USD options are quoted OTC in delta-space (ATM, 25-delta, 10-delta risk reversals and butterflies) rather than strike-space. The smile is reconstructed at each tenor following Clark (2011), eqs. 3.15-3.16:

- σ(call 25Δ) = ATM + BF₂₅ + 0.5·RR₂₅
- σ(put 25Δ)  = ATM + BF₂₅ − 0.5·RR₂₅
- σ(call 10Δ) = ATM + BF₁₀ + 0.5·RR₁₀
- σ(put 10Δ)  = ATM + BF₁₀ − 0.5·RR₁₀

Strikes are computed from delta via Garman-Kohlhagen forward delta inversion (Castagna 2010, sec. 3.8.1; treated as forward deltas, Clark 2011, sec. 3.3.2). The resulting five points per tenor are linearly interpolated (Delaunay) onto an EUR/USD grid of 13 moneyness nodes (0.85-1.15) × 7 tenors (30-730 days). The SPX and EUR/USD grids differ; each surface only builds its own marginal in Section 9.


In [ ]:
# ============================================================
# Cell 7.1 - EUR/USD vol surface from LSEG OTC composite
# Input: Raw_Data/eurusd_vol_snapshots_74.csv
# Output: EUR_vol_surface_grid.csv + sample 3D plot
# ============================================================

LSEG_VOL = RAWDATA / 'eurusd_vol_snapshots_74.csv'
if not LSEG_VOL.exists():
    raise FileNotFoundError(f"{LSEG_VOL.name} not found in Raw_Data/")

TENOR_DAYS  = {'1M':30, '2M':60, '3M':90, '6M':180, '9M':270, '1Y':365, '2Y':730}
TENOR_YEARS = {k: v/365.0 for k, v in TENOR_DAYS.items()}

EUR_MONEYNESS_GRID = np.arange(0.85, 1.16, 0.025)
EUR_TTM_GRID_DAYS  = np.array([30, 60, 90, 180, 270, 365, 730])
EUR_TTM_GRID       = EUR_TTM_GRID_DAYS / 365.0

print(f"Loading LSEG vol composite: {LSEG_VOL.name}")
df_lseg = pd.read_csv(LSEG_VOL, header=[0, 1], index_col=0, parse_dates=True)
print(f"  Snapshot dates : {len(df_lseg)}")
print(f"  Total columns  : {df_lseg.shape[1]}")

bid = df_lseg.xs('BID', level=1, axis=1)
ask = df_lseg.xs('ASK', level=1, axis=1)
mid = (bid + ask) / 2.0
print(f"  Mid quotes     : {mid.shape[1]} RICs\n")

# Supporting data
spot_df  = pd.read_csv(INTERMED / 'EURUSD_daily_spot.csv', index_col=0, parse_dates=True)
rates_df = pd.read_csv(INTERMED / 'RATES_lookup.csv', index_col=0, parse_dates=True)
spot_df  = spot_df.ffill()
rates_df = rates_df.ffill()
print(f"  EUR/USD spot   : {len(spot_df)} obs")
print(f"  Rates lookup   : {len(rates_df)} obs\n")

def get_spot(d):
    if d in spot_df.index:
        v = float(spot_df.loc[d, 'EURUSD'])
        if np.isfinite(v): return v
    nearest = spot_df.index[spot_df.index <= d]
    for idx in reversed(nearest):
        v = float(spot_df.loc[idx, 'EURUSD'])
        if np.isfinite(v): return v
    return None

def get_rates_pair(d):
    rows = rates_df.loc[rates_df.index <= d]
    if rows.empty:
        return None, None
    for idx in reversed(rows.index):
        row = rates_df.loc[idx]
        r_d, r_f = float(row['r']), float(row['r_f'])
        if np.isfinite(r_d) and np.isfinite(r_f):
            return r_d, r_f
    return None, None

def safe_get(rcol, d):
    if rcol not in mid.columns:
        return np.nan
    v = mid[rcol].loc[d]
    if isinstance(v, pd.Series):
        v = v.iloc[0]
    return float(v) if pd.notna(v) else np.nan

def gk_strike_from_delta(F, sigma, T, delta, is_call):
    """Garman-Kohlhagen forward delta inversion (Castagna 2010 sec. 3.8.1; forward deltas, Clark 2011 sec. 3.3.2).
    K_call = F * exp(0.5*sigma^2*T - sigma*sqrt(T)*N^{-1}(delta))
    K_put  = F * exp(0.5*sigma^2*T + sigma*sqrt(T)*N^{-1}(delta))
    """
    z = norm.ppf(delta)
    sign = -1.0 if is_call else 1.0
    return F * np.exp(0.5 * sigma**2 * T + sign * sigma * np.sqrt(T) * z)

# Smile reconstruction per (snapshot, tenor)
all_points = []
n_skipped = 0
for d in mid.index:
    S = get_spot(d)
    r_d, r_f = get_rates_pair(d)
    if S is None or r_d is None:
        n_skipped += 1
        continue

    for tenor, T in TENOR_YEARS.items():
        atm  = safe_get(f'EUR{tenor}O=',  d) / 100.0
        rr25 = safe_get(f'EUR{tenor}RR=', d) / 100.0
        bf25 = safe_get(f'EUR{tenor}BF=', d) / 100.0
        rr10 = safe_get(f'EUR{tenor}R10=', d) / 100.0
        bf10 = safe_get(f'EUR{tenor}B10=', d) / 100.0
        if any(np.isnan([atm, rr25, bf25, rr10, bf10])):
            continue

        # Clark (2011) eqs. 3.15-3.16 (butterfly used as smile strangle)
        sig_atm = atm
        sig_c25 = atm + bf25 + 0.5 * rr25
        sig_p25 = atm + bf25 - 0.5 * rr25
        sig_c10 = atm + bf10 + 0.5 * rr10
        sig_p10 = atm + bf10 - 0.5 * rr10
        if min(sig_atm, sig_c25, sig_p25, sig_c10, sig_p10) <= 0:
            continue

        # Forward and 5 strike points
        F = S * np.exp((r_d - r_f) * T)
        K_atm = F * np.exp(0.5 * sig_atm**2 * T)
        K_c25 = gk_strike_from_delta(F, sig_c25, T, 0.25, is_call=True)
        K_p25 = gk_strike_from_delta(F, sig_p25, T, 0.25, is_call=False)
        K_c10 = gk_strike_from_delta(F, sig_c10, T, 0.10, is_call=True)
        K_p10 = gk_strike_from_delta(F, sig_p10, T, 0.10, is_call=False)

        for K, sig in [(K_p10, sig_p10), (K_p25, sig_p25),
                       (K_atm, sig_atm),
                       (K_c25, sig_c25), (K_c10, sig_c10)]:
            m = K / S
            if np.isfinite(m) and np.isfinite(sig):
                all_points.append({'date': d, 'moneyness': m,
                                    'ttm_years': T, 'sigma': sig})

points_df = pd.DataFrame(all_points)
points_df = points_df.dropna(subset=['moneyness','ttm_years','sigma'])
print(f"Smile points: {len(points_df):,} from {points_df['date'].nunique()} snapshots")
print(f"Skipped (missing spot or rates): {n_skipped}\n")

# Linear interpolation (Delaunay) onto the EUR/USD grid
surf_records = []
for d in points_df['date'].unique():
    sub = points_df[points_df.date == d]
    if len(sub) < 10:
        continue
    pts = sub[['moneyness', 'ttm_years']].values
    sigmas = sub['sigma'].values
    gK, gT = np.meshgrid(EUR_MONEYNESS_GRID, EUR_TTM_GRID)
    grid_iv = griddata(pts, sigmas, (gK, gT), method='linear')
    for i, ttm in enumerate(EUR_TTM_GRID):
        for j, m in enumerate(EUR_MONEYNESS_GRID):
            v = grid_iv[i, j]
            if not np.isnan(v):
                surf_records.append({
                    'date': d.strftime('%Y-%m-%d'),
                    'ttm_years': ttm, 'moneyness': m, 'impl_vol': v,
                })

eur_surface_df = pd.DataFrame(surf_records)
eur_surface_df.to_csv(INTERMED / 'EUR_vol_surface_grid.csv', index=False)
print(f"EUR/USD surfaces : {eur_surface_df['date'].nunique()} dates")
print(f"Grid records     : {len(eur_surface_df):,}")
print(f"Saved            : EUR_vol_surface_grid.csv")

# Sample 3D plot for COVID peak
target_d = '2020-03-16'
if target_d in set(eur_surface_df['date']):
    sub = eur_surface_df[eur_surface_df.date == target_d]
    pivot = sub.pivot(index='ttm_years', columns='moneyness', values='impl_vol')
    fig = plt.figure(figsize=(12, 5))
    ax = fig.add_subplot(111, projection='3d')
    gM, gT = np.meshgrid(pivot.columns, pivot.index)
    ax.plot_surface(gM*100, gT*365, pivot.values*100, cmap='Reds',
                    alpha=0.85, edgecolor='none')
    ax.set_xlabel('Moneyness K/S (%)')
    ax.set_ylabel('TTM (days)')
    ax.set_zlabel('IV (%)')
    ax.set_title('EUR/USD Implied Volatility Surface -- 2020-03-16 (COVID peak)')
    plt.savefig(FIGS / 'eur_vol_surface_2020-03-16.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Sample 3D plot   : eur_vol_surface_2020-03-16.png")


## Section 8. Copula calibration and robustness

* **Cell 8.0** - library used by Sections 8-10: SVI/Breeden-Litzenberger marginals and the copula toolkit (log-densities, CDFs, samplers driven by explicit uniforms).
* **Cell 8.1** - Canonical Maximum Likelihood on rank pseudo-observations (Cherubini, Luciano and Vecchiato 2004, sec. 5.4) for Gaussian, Frank, Clayton, Gumbel and Student-t; regime sub-samples tagged by the *previous* day's Cboe VIX close; moving-block bootstrap CIs for regime Kendall tau.
* **Cell 8.2** - robustness: AR(1)-GARCH(1,1)-t filtered residuals, survival Clayton/Gumbel, Cramér-von Mises goodness of fit with parametric bootstrap (Genest, Rémillard and Beaudoin 2009), horizon aggregation check, non-overlapping monthly returns.
* **Cell 8.3** - asymmetric two-parameter families (BB7 and symmetrised Joe-Clayton) and a split-sample out-of-sample comparison.

Note: with one parameter each, the four single-parameter families are ranked identically by AIC and BIC; BIC is informative only against the two-parameter Student-t copula.


In [ ]:
# ============================================================
# Cell 8.0 - Library: option-implied marginals (SVI + Breeden-Litzenberger) and copula toolkit
# Every function below is used by Sections 8-11. Run this cell before Cell 8.1.
# ============================================================
import numpy as np, pandas as pd, json, pickle
from scipy import stats
from scipy.stats import norm
from scipy.optimize import least_squares, minimize_scalar, minimize
from scipy.integrate import quad
"""
Option-implied risk-neutral marginals from the gridded implied-volatility surfaces.

Steps (per asset, snapshot date, target maturity T):
  1. For every maturity pillar of the surface, convert spot-moneyness m = K/S0 into
     forward log-moneyness k = ln(K/F) = ln(m) - (r - y) * T_pillar   (y = q for SPX, r_f for EUR/USD).
  2. Fit a raw-SVI total-variance slice w(k) = a + b*(rho*(k-m) + sqrt((k-m)^2 + s^2))
     (Gatheral, 2006; Gatheral & Jacquier, 2014) to the pillar's total variance sigma^2 * T.
  3. Interpolate total variance linearly in T between the two bracketing pillars at fixed k.
     Beyond the last pillar the implied volatility at fixed k is held flat (w scales with T).
  4. Risk-neutral survival function (Breeden & Litzenberger, 1978), written in total variance:
         P(S_T > K) = N(d2) - phi(d2) * d sqrt(w)/dk,   d2 = -k/sqrt(w) - sqrt(w)/2
     CDF = 1 - P(S_T > K). Negative density (butterfly arbitrage) is recorded and removed by
     enforcing monotonicity; the inverse CDF is then tabulated.
  5. Martingale check: E[S_T]/F computed on the tabulated law; draws are rescaled so that E[S_T] = F.
"""


# ---------------------------------------------------------------- SVI slice
def svi_w(k, p):
    a, b, rho, m, s = p
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + s ** 2))


def svi_dw(k, p):
    a, b, rho, m, s = p
    return b * (rho + (k - m) / np.sqrt((k - m) ** 2 + s ** 2))


def svi_d2w(k, p):
    a, b, rho, m, s = p
    return b * s ** 2 / ((k - m) ** 2 + s ** 2) ** 1.5


def svi_g(k, p):
    """Gatheral-Jacquier (2014) butterfly function; density >= 0 iff g(k) >= 0."""
    w = np.maximum(svi_w(k, p), 1e-12); w1 = svi_dw(k, p); w2 = svi_d2w(k, p)
    return (1 - k * w1 / (2 * w)) ** 2 - w1 ** 2 / 4 * (1 / w + 0.25) + w2 / 2


def fit_svi(k, w, w_floor=None, k_floor=None):
    """Raw-SVI least-squares fit with no-arbitrage penalties:
       Lee (2004) wing bound b(1+|rho|) <= 2, w >= 0, butterfly g(k) >= 0 on a wide k grid,
       and (if w_floor given) no calendar arbitrage: w(k) >= w_floor(k) of the previous pillar."""
    k = np.asarray(k, float); w = np.asarray(w, float)
    wmin, wmax = w.min(), w.max()
    span = max(np.ptp(k), 0.05)
    ext = max(1.5 * span, 6.0 * np.sqrt(wmax))
    kg = np.linspace(k.min() - ext, k.max() + ext, 161)
    best = None
    for rho0 in (-0.7, -0.3, 0.0, 0.3):
        for s0 in (0.05, 0.2):
            x0 = [max(wmin * 0.8, 1e-6), max((wmax - wmin) / span, 0.01), rho0,
                  float(k[np.argmin(w)]), s0]
            lb = [-wmax, 0.0, -0.999, k.min() - 1.0, 1e-3]
            ub = [wmax * 2 + 1e-4, 2.0, 0.999, k.max() + 1.0, 2.0]
            x0 = np.clip(x0, np.array(lb) + 1e-9, np.array(ub) - 1e-9)

            def res(p):
                a, b, rho, m, s = p
                r_fit = (svi_w(k, p) - w) / max(wmax, 1e-8)
                pen_lee = max(0.0, b * (1 + abs(rho)) - 2.0) * 10
                pen_pos = max(0.0, -(a + b * s * np.sqrt(1 - rho ** 2))) * 10
                pen_bf = np.minimum(svi_g(kg, p), 0.0) * 2.0
                out = [r_fit, [pen_lee, pen_pos], pen_bf]
                if w_floor is not None:
                    out.append(np.minimum(svi_w(k_floor, p) - w_floor, 0.0) / max(wmax, 1e-8) * 5)
                return np.concatenate(out)

            try:
                sol = least_squares(res, x0, bounds=(lb, ub), max_nfev=3000)
            except Exception:
                continue
            if best is None or sol.cost < best.cost:
                best = sol
    if best is None:
        raise ValueError('SVI fit failed for every starting point (check for NaN inputs: rates, moneyness or vols)')
    p = best.x
    rmse_vol = float(np.sqrt(np.mean((np.sqrt(np.maximum(svi_w(k, p), 1e-12)) - np.sqrt(w)) ** 2)))
    return p, rmse_vol


# ---------------------------------------------------------------- surface object
class SmileSurface:
    """Holds SVI slices for one asset on one snapshot date."""

    def __init__(self, grid_df, r, y, min_points=4):
        # grid_df: rows (ttm_years, moneyness, impl_vol) for one date
        self.slices = []  # (T_pillar, params, rmse_vol_bp, n_points, k_min, k_max)
        prev = None
        for T, g in sorted(grid_df.groupby('ttm_years'), key=lambda x: x[0]):
            g = g.dropna(subset=['impl_vol'])
            if len(g) < min_points:
                continue
            k = np.log(g['moneyness'].values) - (r - y) * T
            w = g['impl_vol'].values ** 2 * T
            if prev is None:
                p, rm = fit_svi(k, w)
            else:
                kf = np.linspace(min(k.min(), prev[4]) - 0.2, max(k.max(), prev[5]) + 0.2, 61)
                p, rm = fit_svi(k, w, w_floor=svi_w(kf, prev[1]), k_floor=kf)
            prev = (float(T), p, rm, len(g), float(k.min()), float(k.max()))
            self.slices.append(prev)
        self.slices.sort(key=lambda s: s[0])
        self.Ts = np.array([s[0] for s in self.slices])

    def to_list(self):
        return [[s[0], [float(x) for x in s[1]], s[2], s[3], s[4], s[5]] for s in self.slices]

    @classmethod
    def from_list(cls, lst):
        obj = cls.__new__(cls)
        obj.slices = [(float(a), np.array(b), c, d, e, f) for a, b, c, d, e, f in lst]
        obj.Ts = np.array([s[0] for s in obj.slices])
        return obj

    def w_and_dw(self, k, T):
        Ts = self.Ts
        if T >= Ts[-1]:
            _, p, *_ = self.slices[-1]
            # Beyond the last pillar: ATM implied vol held flat and the smile kept fixed in
            # standardised moneyness k/sqrt(w_ATM):  w_T(k) = f * w_L(k / sqrt(f)),  f = T / T_L
            f = T / Ts[-1]
            x = k / np.sqrt(f)
            return svi_w(x, p) * f, svi_dw(x, p) * np.sqrt(f), ('extrap', Ts[-1])
        if T <= Ts[0]:
            _, p, *_ = self.slices[0]
            f = T / Ts[0]
            return svi_w(k, p) * f, svi_dw(k, p) * f, ('extrap', Ts[0])
        j = int(np.searchsorted(Ts, T)); i = j - 1
        if abs(Ts[j] - T) < 1e-9:
            _, p, *_ = self.slices[j]
            return svi_w(k, p), svi_dw(k, p), ('pillar', Ts[j])
        lam = (T - Ts[i]) / (Ts[j] - Ts[i])
        pi, pj = self.slices[i][1], self.slices[j][1]
        w = (1 - lam) * svi_w(k, pi) + lam * svi_w(k, pj)
        dw = (1 - lam) * svi_dw(k, pi) + lam * svi_dw(k, pj)
        return w, dw, ('interp', (Ts[i], Ts[j]))

    def atm_vol(self, T):
        w, _, src = self.w_and_dw(np.array([0.0]), T)
        return float(np.sqrt(max(w[0], 1e-12) / T)), src

    def butterfly_min_g(self, kgrid):
        return min(float(svi_g(kgrid, s[1]).min()) for s in self.slices)

    def calendar_violations(self, kgrid):
        """Count pillar pairs where total variance decreases in T at some k (calendar arbitrage)."""
        bad = 0
        for (T1, p1, *_), (T2, p2, *_) in zip(self.slices[:-1], self.slices[1:]):
            if np.any(svi_w(kgrid, p2) < svi_w(kgrid, p1) - 1e-10):
                bad += 1
        return bad


class ImpliedMarginal:
    """Tabulated risk-neutral law of S_T / S_0 from a SmileSurface at maturity T."""

    def __init__(self, surf, T, fwd_ratio, n_grid=6001):
        self.T = T
        w0, _, self.src = surf.w_and_dw(np.array([0.0]), T)
        sd = np.sqrt(max(w0[0], 1e-8))
        k = np.linspace(-9 * sd, 9 * sd, n_grid)
        w, dw, _ = surf.w_and_dw(k, T)
        w = np.maximum(w, 1e-10)
        sw = np.sqrt(w)
        d2 = -k / sw - sw / 2
        surv = norm.cdf(d2) - norm.pdf(d2) * dw / (2 * sw)
        cdf = 1.0 - surv
        dens = np.diff(cdf)
        self.neg_mass = float(-dens[dens < 0].sum())      # butterfly-arbitrage diagnostic
        cdf = np.maximum.accumulate(np.clip(cdf, 0.0, 1.0))
        # keep strictly increasing support for inversion
        keep = np.r_[True, np.diff(cdf) > 1e-14]
        k, cdf = k[keep], cdf[keep]
        self.tail_lo, self.tail_hi = float(cdf[0]), float(1 - cdf[-1])
        self.k, self.cdf = k, cdf
        # martingale check on tabulated law (mid-point rule on CDF increments)
        km = 0.5 * (k[1:] + k[:-1]); pm = np.diff(cdf)
        mass = pm.sum()
        self.mean_ratio = float((np.exp(km) * pm).sum() / mass)   # E[S_T]/F before correction
        self.scale = fwd_ratio / self.mean_ratio                    # so that E[S_T/S0] = F/S0
        self.atm_vol = float(sd / np.sqrt(T))

    def ppf(self, u):
        # rescale u into tabulated range then invert
        u = self.cdf[0] + u * (self.cdf[-1] - self.cdf[0])
        return self.scale * np.exp(np.interp(u, self.cdf, self.k))


class LognormalMarginal:
    """Flat-smile benchmark: lognormal at the ATM-forward implied volatility of the same surface."""

    def __init__(self, sigma, T, fwd_ratio):
        self.sigma, self.T, self.fwd = sigma, T, fwd_ratio

    def ppf(self, u):
        z = norm.ppf(np.clip(u, 1e-12, 1 - 1e-12))
        return self.fwd * np.exp(-0.5 * self.sigma ** 2 * self.T + self.sigma * np.sqrt(self.T) * z)


"""
Copula families: log-densities (for CML), CDFs (for validation), and samplers driven by
explicit independent uniforms so that antithetic variates are applied to the DRIVING uniforms
(every simulated pair, original and antithetic, is a draw from the intended copula).
"""

FAMILIES = ['gaussian', 'frank', 'clayton', 'gumbel']      # proposal families
PRICING_FAMILIES = FAMILIES + ['student_t']                # t-copula added as tail-dependent symmetric benchmark
N_DRIVERS = {'gaussian': 2, 'frank': 2, 'clayton': 2, 'gumbel': 4, 'student_t': 3}
EPS = 1e-12


# ------------------------------------------------------------------ log-densities
def ll_gaussian(rho, u, v):
    if abs(rho) >= 0.999: return -1e10
    x, y = stats.norm.ppf(u), stats.norm.ppf(v)
    return float(np.sum(-0.5 * np.log(1 - rho ** 2) - (rho ** 2 * (x ** 2 + y ** 2) - 2 * rho * x * y) / (2 * (1 - rho ** 2))))


def ll_frank(t, u, v):
    if abs(t) < 1e-8: return 0.0
    g0 = 1 - np.exp(-t); gu = 1 - np.exp(-t * u); gv = 1 - np.exp(-t * v)
    num = t * g0 * np.exp(-t * (u + v)); den = (g0 - gu * gv) ** 2
    return float(np.sum(np.log(np.maximum(num / den, 1e-300))))


def ll_clayton(t, u, v):
    if t <= 0: return -1e10
    inner = u ** (-t) + v ** (-t) - 1
    return float(np.sum(np.log(1 + t) + (-2 - 1 / t) * np.log(inner) + (-1 - t) * (np.log(u) + np.log(v))))


def ll_gumbel(d, u, v):
    if d < 1: return -1e10
    if abs(d - 1) < 1e-10: return 0.0
    lu, lv = -np.log(u), -np.log(v)
    A = lu ** d + lv ** d; Ai = A ** (1 / d)
    lc = (-Ai + lu + lv + (d - 1) * (np.log(lu) + np.log(lv)) + (1 / d - 2) * np.log(A) + np.log(Ai + d - 1))
    return float(np.sum(lc))


def ll_t(params, u, v):
    rho, nu = params
    if abs(rho) >= 0.999 or nu <= 2.0 or nu > 200: return -1e10
    x, y = stats.t.ppf(u, nu), stats.t.ppf(v, nu)
    z = (x ** 2 + y ** 2 - 2 * rho * x * y) / (nu * (1 - rho ** 2))
    from scipy.special import gammaln
    lc = (gammaln((nu + 2) / 2) + gammaln(nu / 2) - 2 * gammaln((nu + 1) / 2) - 0.5 * np.log(1 - rho ** 2)
          - (nu + 2) / 2 * np.log1p(z) + (nu + 1) / 2 * (np.log1p(x ** 2 / nu) + np.log1p(y ** 2 / nu)))
    return float(np.sum(lc))


LL = {'gaussian': ll_gaussian, 'frank': ll_frank, 'clayton': ll_clayton, 'gumbel': ll_gumbel,
      # survival (180-degree rotated) versions: c_surv(u,v) = c(1-u,1-v)
      'surv_clayton': lambda t, u, v: ll_clayton(t, 1 - u, 1 - v),
      'surv_gumbel': lambda d, u, v: ll_gumbel(d, 1 - u, 1 - v)}
BOUNDS = {'gaussian': (-0.99, 0.99), 'frank': (-30, 30), 'clayton': (1e-4, 15), 'gumbel': (1.0001, 10),
          'surv_clayton': (1e-4, 15), 'surv_gumbel': (1.0001, 10)}


def pseudo_obs(x):
    return stats.rankdata(x) / (len(x) + 1.0)


def fit_family(name, u, v):
    if name == 'student_t':
        best = None
        for nu0 in (4, 8, 20):
            r0 = np.sin(np.pi / 2 * stats.kendalltau(u, v)[0])
            res = minimize(lambda p: -ll_t(p, u, v), x0=[r0, nu0], method='Nelder-Mead',
                           options={'xatol': 1e-6, 'fatol': 1e-8, 'maxiter': 4000})
            if best is None or res.fun < best.fun: best = res
        return {'param': [float(best.x[0]), float(best.x[1])], 'll': -float(best.fun), 'k': 2}
    res = minimize_scalar(lambda t: -LL[name](t, u, v), bounds=BOUNDS[name], method='bounded',
                          options={'xatol': 1e-7})
    return {'param': float(res.x), 'll': -float(res.fun), 'k': 1}


def debye1(t):
    return quad(lambda x: x / np.expm1(x), 0, t)[0] / t if abs(t) > 1e-8 else 1.0


def implied_tau(name, p):
    if name in ('gaussian',): return 2 / np.pi * np.arcsin(p)
    if name == 'student_t': return 2 / np.pi * np.arcsin(p[0])
    if name in ('clayton', 'surv_clayton'): return p / (p + 2)
    if name in ('gumbel', 'surv_gumbel'): return 1 - 1 / p
    if name == 'frank':
        # tau = 1 - (4/theta) * (1 - D1(theta)),  D1 = Debye function of order 1
        return 1 - 4 / p * (1 - debye1(p)) if p > 0 else -(1 - 4 / abs(p) * (1 - debye1(abs(p))))


def tail_dep(name, p):
    """(lambda_L, lambda_U)"""
    if name == 'clayton': return 2 ** (-1 / p), 0.0
    if name == 'surv_clayton': return 0.0, 2 ** (-1 / p)
    if name == 'gumbel': return 0.0, 2 - 2 ** (1 / p)
    if name == 'surv_gumbel': return 2 - 2 ** (1 / p), 0.0
    if name == 'student_t':
        r, nu = p; lam = 2 * stats.t.cdf(-np.sqrt((nu + 1) * (1 - r) / (1 + r)), nu + 1); return lam, lam
    return 0.0, 0.0


# ------------------------------------------------------------------ copula CDFs (validation)
def cdf(name, p, u, v):
    if name == 'gaussian':
        return stats.multivariate_normal(mean=[0, 0], cov=[[1, p], [p, 1]]).cdf(np.c_[stats.norm.ppf(u), stats.norm.ppf(v)])
    if name == 'frank':
        return -1 / p * np.log(1 + np.expm1(-p * u) * np.expm1(-p * v) / np.expm1(-p))
    if name == 'clayton':
        return np.maximum(u ** (-p) + v ** (-p) - 1, 0) ** (-1 / p)
    if name == 'gumbel':
        return np.exp(-((-np.log(u)) ** p + (-np.log(v)) ** p) ** (1 / p))


# ------------------------------------------------------------------ samplers (driven)
def sample_driven(name, p, W):
    """W: (n, N_DRIVERS[name]) independent U(0,1) drivers.  Returns (u, v) with copula `name`."""
    W = np.clip(W, 1e-12, 1 - 1e-12)
    a, b = W[:, 0], W[:, 1]
    if name == 'gaussian':
        z1 = stats.norm.ppf(a); z2 = p * z1 + np.sqrt(1 - p ** 2) * stats.norm.ppf(b)
        return a, stats.norm.cdf(z2)
    if name == 'frank':                       # conditional inversion, Cherubini et al. (2004) pp.186-187
        v = -np.log(1 + b * np.expm1(-p) / (b + (1 - b) * np.exp(-p * a))) / p
        return a, np.clip(v, EPS, 1 - EPS)
    if name == 'clayton':                     # conditional inversion, Cherubini et al. (2004) pp.184-185
        v = a * (b ** (-p / (1 + p)) - 1 + a ** p) ** (-1 / p)
        return a, np.clip(v, EPS, 1 - EPS)
    if name == 'gumbel':                      # Marshall-Olkin frailty; positive alpha-stable via
        al = 1 / p                             # Chambers-Mallows-Stuck (1976) / Kanter representation
        U = np.pi * a; Wexp = -np.log(b); E1 = -np.log(W[:, 2]); E2 = -np.log(W[:, 3])
        M = (np.sin(al * U) / np.sin(U) ** (1 / al)) * (np.sin((1 - al) * U) / Wexp) ** ((1 - al) / al)
        return np.clip(np.exp(-(E1 / M) ** al), EPS, 1 - EPS), np.clip(np.exp(-(E2 / M) ** al), EPS, 1 - EPS)
    if name == 'student_t':                   # p = (rho, nu): normal mixture with chi-square(nu) driver
        rho, nu = p
        z1 = stats.norm.ppf(a); z2 = rho * z1 + np.sqrt(1 - rho ** 2) * stats.norm.ppf(b)
        w = np.sqrt(nu / stats.chi2.ppf(W[:, 2], nu))
        return stats.t.cdf(z1 * w, nu), stats.t.cdf(z2 * w, nu)
    raise ValueError(name)


def sample_antithetic(name, p, W):
    """Returns (u, v) of length 2n: first n from drivers W, next n from 1 - W (antithetic drivers)."""
    u1, v1 = sample_driven(name, p, W)
    u2, v2 = sample_driven(name, p, 1 - W)
    return np.r_[u1, u2], np.r_[v1, v2]


def save_surfaces(surfaces, path):
    out = {d: {k: (v.to_list() if isinstance(v, SmileSurface) else v) for k, v in s.items()} for d, s in surfaces.items()}
    with open(path, 'w') as f: json.dump(out, f)


def load_surfaces(path):
    raw = json.load(open(path))
    return {d: {k: (SmileSurface.from_list(v) if k in ('spx', 'eur') else v) for k, v in s.items()} for d, s in raw.items()}


In [ ]:
# ============================================================
# Cell 8.1 - Copula calibration (CML on rank pseudo-observations)
#   * full sample: Gaussian, Frank, Clayton, Gumbel (+ Student-t)
#   * regime sub-samples: each daily return is tagged with the PREVIOUS trading day's
#     Cboe VIX close (information available before the return is realised; same-day
#     tagging is kept for comparison only)
#   * moving-block bootstrap (block 20 days, 2,000 reps) for regime Kendall tau
# Output: calibration.json, copula_params.json (full sample), Table1_copula_calibration.csv
# ============================================================
pr = pd.read_csv(INTERMED / 'PAIRED_returns_copula.csv', index_col=0, parse_dates=True)
vix_s = pd.read_csv(INTERMED / 'VIX_daily.csv', index_col=0, parse_dates=True)['Close']
regime_of = lambda x: np.where(x < 20, 'calm', np.where(x < 30, 'elevated', 'stress'))
pr['vix_prev'] = vix_s.shift(1).reindex(pr.index, method='ffill').values
pr['vix_same'] = vix_s.reindex(pr.index, method='ffill').values
pr['reg_prev'] = regime_of(pr.vix_prev); pr['reg_same'] = regime_of(pr.vix_same)
x_ret, y_ret = pr.SPX_LogReturn.values, pr.EUR_LogReturn.values

def calibrate(x, y, fams=FAMILIES + ['student_t']):
    u, v = pseudo_obs(x), pseudo_obs(y); n = len(u)
    out = {'n': n, 'kendall_tau': float(stats.kendalltau(x, y)[0]),
           'pearson': float(np.corrcoef(x, y)[0, 1]), 'spearman': float(stats.spearmanr(x, y)[0])}
    for f in fams:
        r = fit_family(f, u, v); r['aic'] = 2 * r['k'] - 2 * r['ll']; r['bic'] = r['k'] * np.log(n) - 2 * r['ll']
        r['implied_tau'] = float(implied_tau(f, r['param'])); r['lambda_L'], r['lambda_U'] = map(float, tail_dep(f, r['param']))
        out[f] = r
    return out

calib = {'full': calibrate(x_ret, y_ret)}
for tag in ['prev', 'same']:
    for g in ['calm', 'elevated', 'stress']:
        m = pr['reg_' + tag] == g
        calib[f'{g}_{tag}'] = calibrate(x_ret[m], y_ret[m])
for k, r in calib.items():
    print(f"{k:14s} n={r['n']:5d} tau={r['kendall_tau']:.4f} | " +
          ' '.join(f"{f}={np.round(r[f]['param'], 4)} (AIC {r[f]['aic']:.2f})" for f in FAMILIES + ['student_t']))

rng_b = np.random.default_rng(11); n_obs = len(pr); L = 20; B = 2000
regs = pr.reg_prev.values; taus = {g: [] for g in ['calm', 'elevated', 'stress']}
for b in range(B):
    st = rng_b.integers(0, n_obs - L, int(np.ceil(n_obs / L)))
    idx = np.concatenate([np.arange(s, s + L) for s in st])[:n_obs]
    xs, ys, rs = x_ret[idx], y_ret[idx], regs[idx]
    for g in taus: taus[g].append(stats.kendalltau(xs[rs == g], ys[rs == g])[0])
ci = {g: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for g, v in taus.items()}
ci['calm_minus_stress'] = list(np.percentile(np.array(taus['calm']) - np.array(taus['stress']), [2.5, 97.5]))
ci['elevated_minus_calm'] = list(np.percentile(np.array(taus['elevated']) - np.array(taus['calm']), [2.5, 97.5]))
calib['bootstrap_tau_ci'] = ci
print('Block-bootstrap 95% CIs for regime Kendall tau:', json.dumps(ci, default=float))

with open(INTERMED / 'calibration.json', 'w') as f: json.dump(calib, f, indent=1, default=float)
with open(INTERMED / 'copula_params.json', 'w') as f: json.dump(calib['full'], f, indent=1, default=float)
t1 = pd.DataFrame({f: {k: calib['full'][f][k] for k in ['param', 'll', 'aic', 'bic', 'implied_tau', 'lambda_L', 'lambda_U']}
                   for f in FAMILIES + ['student_t']}).T
t1.to_csv(RESULTS / 'Table1_copula_calibration.csv'); print(t1)
pr.to_csv(INTERMED / 'PAIRED_returns_with_regimes.csv')


In [ ]:
# ============================================================
# Cell 8.2 - Copula robustness
#   1. AR(1)-GARCH(1,1)-t filter on each return series; refit on standardised residuals
#   2. Extra families: Student-t, survival Clayton, survival Gumbel
#   3. Cramer-von Mises goodness-of-fit test with parametric bootstrap
#      (Genest, Remillard and Beaudoin, 2009)
#   4. Horizon check: copula of h-day sums when daily draws follow the fitted Gumbel
#   5. Non-overlapping 21-day returns
# Output: copula_robustness.json   (runtime ~10 minutes, the t-copula GoF dominates)
# ============================================================
!pip install arch -q
import time
from arch import arch_model

pr = pd.read_csv(INTERMED / 'PAIRED_returns_copula.csv', index_col=0, parse_dates=True)
x, y = pr.SPX_LogReturn.values, pr.EUR_LogReturn.values
FAMS7 = ['gaussian', 'student_t', 'frank', 'clayton', 'gumbel', 'surv_clayton', 'surv_gumbel']
out = {}

# ---------------------------------------------------------------- 1. GARCH filter
def std_resid(r):
    am = arch_model(r * 100, mean='AR', lags=1, vol='GARCH', p=1, q=1, dist='t')
    res = am.fit(disp='off')
    z = res.std_resid
    return np.asarray(z)[1:], res  # first obs lost to AR(1)

zx, rx = std_resid(x); zy, ry = std_resid(y)
out['garch_params'] = {'spx': {k: float(v) for k, v in rx.params.items()}, 'eur': {k: float(v) for k, v in ry.params.items()}}
# Ljung-Box on squared standardised residuals (lag 10) before / after filtering
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = lambda s: float(acorr_ljungbox(s ** 2, lags=[10])['lb_pvalue'].iloc[0])
out['ljungbox_sq_p'] = {'spx_raw': lb(x - x.mean()), 'spx_filtered': lb(zx), 'eur_raw': lb(y - y.mean()), 'eur_filtered': lb(zy)}

def fit_table(a, b):
    u, v = pseudo_obs(a), pseudo_obs(b); n = len(u); tab = {}
    for f in FAMS7:
        r = fit_family(f, u, v); r['aic'] = 2 * r['k'] - 2 * r['ll']; r['bic'] = r['k'] * np.log(n) - 2 * r['ll']
        r['lambda_L'], r['lambda_U'] = map(float, tail_dep(f, r['param'])); tab[f] = r
    tab['n'] = n; tab['kendall_tau'] = float(stats.kendalltau(a, b)[0])
    return tab, u, v

out['raw'], u_raw, v_raw = fit_table(x, y)
out['garch'], u_g, v_g = fit_table(zx, zy)
for k in ('raw', 'garch'):
    print(k, 'tau', round(out[k]['kendall_tau'], 4))
    for f in FAMS7:
        r = out[k][f]; print(f"   {f:13s} param={np.round(r['param'],4)}  LL={r['ll']:.2f} AIC={r['aic']:.2f} BIC={r['bic']:.2f} lL={r['lambda_L']:.3f} lU={r['lambda_U']:.3f}")

# ---------------------------------------------------------------- 2. Cramer-von Mises GoF (Genest, Remillard & Beaudoin, 2009)
def emp_cop(u, v):
    return np.array([np.mean((u <= ui) & (v <= vi)) for ui, vi in zip(u, v)])

def cdf_any(f, p, u, v):
    if f == 'surv_clayton': return u + v - 1 + cdf('clayton', p, 1 - u, 1 - v)
    if f == 'surv_gumbel': return u + v - 1 + cdf('gumbel', p, 1 - u, 1 - v)
    if f == 'student_t':
        # t-copula CDF evaluated from a 1,000,000-draw simulated grid (400 x 400 cells, bilinear);
        # simulation error ~5e-4 per point, negligible relative to S_n
        r, nu = p
        a, b = sample_any('student_t', p, 1_000_000, np.random.default_rng(12345))
        H, _, _ = np.histogram2d(a, b, bins=400, range=[[0, 1], [0, 1]])
        G = np.zeros((401, 401)); G[1:, 1:] = H.cumsum(0).cumsum(1) / len(a)
        from scipy.interpolate import RegularGridInterpolator
        g = np.linspace(0, 1, 401)
        return RegularGridInterpolator((g, g), G)(np.c_[u, v])
    return cdf(f, p, u, v)

def sample_any(f, p, n, rng):
    if f in ('surv_clayton', 'surv_gumbel'):
        a, b = sample_driven(f[5:], p, rng.random((n, N_DRIVERS[f[5:]]))); return 1 - a, 1 - b
    if f == 'student_t':
        r, nu = p; z = rng.multivariate_normal([0, 0], [[1, r], [r, 1]], n); w = np.sqrt(nu / rng.chisquare(nu, n))
        t = z * w[:, None]; return stats.t.cdf(t[:, 0], nu), stats.t.cdf(t[:, 1], nu)
    return sample_driven(f, p, rng.random((n, N_DRIVERS[f])))

def cvm(f, u, v, B, rng):
    p = fit_family(f, u, v)['param']
    Sn = np.sum((emp_cop(u, v) - cdf_any(f, p, u, v)) ** 2)
    Sb = []
    for _ in range(B):
        a, b = sample_any(f, p, len(u), rng); a, b = pseudo_obs(a), pseudo_obs(b)
        pb = fit_family(f, a, b)['param']
        Sb.append(np.sum((emp_cop(a, b) - cdf_any(f, pb, a, b)) ** 2))
    return float(Sn), float((np.sum(np.array(Sb) >= Sn) + 0.5) / (B + 1))

rng = np.random.default_rng(2024)
out['gof'] = {}
for tag, (u, v) in {'raw': (u_raw, v_raw), 'garch': (u_g, v_g)}.items():
    out['gof'][tag] = {}
    for f in ['gaussian', 'frank', 'clayton', 'gumbel', 'surv_clayton', 'student_t']:
        t0 = time.time(); B = 100 if f == 'student_t' else 250
        Sn, pv = cvm(f, u, v, B, rng)
        out['gof'][tag][f] = {'Sn': Sn, 'p_value': pv, 'B': B}
        print(f'GoF {tag:5s} {f:13s} Sn={Sn:.4f} p={pv:.3f} (B={B}, {time.time()-t0:.0f}s)', flush=True)

# ---------------------------------------------------------------- 3. Horizon aggregation (C5)
# If daily (U,V) follow the fitted daily Gumbel copula and daily returns are i.i.d., what copula do h-day sums have?
rng = np.random.default_rng(99); n_paths = 40000; dG = out['raw']['gumbel']['param']
sx, sy = x.std(), y.std()
hz = {}
for h in [1, 5, 21, 63, 126, 252]:
    X = np.zeros(n_paths); Y = np.zeros(n_paths)
    for _ in range(h):
        a, b = sample_driven('gumbel', dG, rng.random((n_paths, 4)))
        X += stats.norm.ppf(a) * sx; Y += stats.norm.ppf(b) * sy
    a, b = pseudo_obs(X), pseudo_obs(Y)
    q = 0.95
    hz[h] = {'kendall_tau': float(stats.kendalltau(a[:8000], b[:8000])[0]),
             'upper_conc_95': float(np.mean((a > q) & (b > q)) / (1 - q)),
             'lower_conc_95': float(np.mean((a < 1 - q) & (b < 1 - q)) / (1 - q))}
    print('horizon', h, hz[h], flush=True)
# Gaussian reference with the same tau
rG = np.sin(np.pi / 2 * hz[1]['kendall_tau'])
a, b = sample_driven('gaussian', rG, rng.random((200000, 2)))
hz['gaussian_same_tau'] = {'upper_conc_95': float(np.mean((a > .95) & (b > .95)) / .05), 'lower_conc_95': float(np.mean((a < .05) & (b < .05)) / .05)}
out['horizon'] = hz
# Non-overlapping 21-day (monthly) returns: does Gumbel still beat Gaussian?
m = pr[['SPX_LogReturn', 'EUR_LogReturn']].groupby(np.arange(len(pr)) // 21).sum()
out['monthly_nonoverlap'], *_ = fit_table(m.SPX_LogReturn.values, m.EUR_LogReturn.values)
print('monthly n', out['monthly_nonoverlap']['n'], {f: round(out['monthly_nonoverlap'][f]['aic'], 2) for f in FAMS7})

json.dump(out, open(INTERMED / 'copula_robustness.json', 'w'), indent=1, default=float)
print('saved')


In [ ]:
# ============================================================
# Cell 8.3 - Two further copula checks (Section 5.2)
#   1. Asymmetric two-parameter families: Joe-Clayton (BB7) and the symmetrised
#      Joe-Clayton (SJC) of Patton (2006); likelihood-ratio test of tau_U = tau_L
#      and a 20-day block bootstrap of tau_U - tau_L
#   2. Out-of-sample ranking: fit on 2018-2020, score on 2021-2022 (and the reverse);
#      per-observation log scores compared with the Gaussian copula (HAC t-statistic)
# Output: copula_extra_checks.json   (runtime ~5 minutes, the bootstrap dominates)
# ============================================================
def _copula_extra_checks():
    import sympy as sp
    from scipy.optimize import minimize
    pr = pd.read_csv(INTERMED / 'PAIRED_returns_copula.csv', index_col=0, parse_dates=True)
    X, Y = pr.SPX_LogReturn.values, pr.EUR_LogReturn.values
    FAMS = ['gaussian', 'frank', 'clayton', 'gumbel', 'student_t']

    # ---------------- BB7 (Joe-Clayton) density from the CDF, symbolic mixed derivative
    us, vs, k, g = sp.symbols('u v k g', positive=True)
    C = 1 - (1 - ((1 - (1 - us) ** k) ** (-g) + (1 - (1 - vs) ** k) ** (-g) - 1) ** (-1 / g)) ** (1 / k)
    c_bb7 = sp.lambdify((us, vs, k, g), sp.diff(C, us, vs), 'numpy')

    def k_g(tU, tL):   # Patton (2006): kappa from upper, gamma from lower tail dependence
        return 1 / np.log2(2 - tU), -1 / np.log2(tL)

    def ll_bb7(p, u, v):
        tU, tL = p
        if not (1e-4 < tU < 0.95 and 1e-4 < tL < 0.95): return -1e10
        kk, gg = k_g(tU, tL)
        with np.errstate(all='ignore'): d = c_bb7(u, v, kk, gg)
        return float(np.sum(np.log(np.maximum(d, 1e-300)))) if np.all(np.isfinite(d)) else -1e10

    def ll_sjc(p, u, v):
        tU, tL = p
        if not (1e-4 < tU < 0.95 and 1e-4 < tL < 0.95): return -1e10
        k1, g1 = k_g(tU, tL); k2, g2 = k_g(tL, tU)
        with np.errstate(all='ignore'):
            d = 0.5 * (c_bb7(u, v, k1, g1) + c_bb7(1 - u, 1 - v, k2, g2))
        return float(np.sum(np.log(np.maximum(d, 1e-300)))) if np.all(np.isfinite(d)) else -1e10

    def fit2(ll, u, v):
        best = None
        for x0 in ([0.1, 0.1], [0.2, 0.05], [0.05, 0.2], [0.3, 0.3]):
            r = minimize(lambda p: -ll(p, u, v), x0, method='Nelder-Mead', options={'xatol': 1e-7, 'fatol': 1e-9, 'maxiter': 4000})
            if best is None or r.fun < best.fun: best = r
        return best

    def fit_sym_sjc(u, v):
        best = None
        for x0 in (0.05, 0.15, 0.3):
            r = minimize(lambda p: -ll_sjc([p[0], p[0]], u, v), [x0], method='Nelder-Mead', options={'xatol': 1e-8, 'fatol': 1e-10})
            if best is None or r.fun < best.fun: best = r
        return best

    out = {}
    u, v = pseudo_obs(X), pseudo_obs(Y); n = len(u)
    B = {}
    for name, ll in (('bb7', ll_bb7), ('sjc', ll_sjc)):
        r = fit2(ll, u, v)
        B[name] = {'tau_U': float(r.x[0]), 'tau_L': float(r.x[1]), 'll': float(-r.fun), 'aic': float(4 + 2 * r.fun)}
    rs = fit_sym_sjc(u, v)
    B['sjc_symmetric'] = {'tau_U=tau_L': float(rs.x[0]), 'll': float(-rs.fun), 'aic': float(2 + 2 * rs.fun)}
    LR = 2 * (B['sjc']['ll'] - B['sjc_symmetric']['ll'])
    B['LR_symmetry'] = {'stat': float(LR), 'p_value': float(stats.chi2.sf(LR, 1))}
    # parametric-free uncertainty: block bootstrap of (tau_U - tau_L) for the SJC
    rng = np.random.default_rng(20261013); L = 20; diffs = []
    for b in range(200):
        starts = rng.integers(0, n - L, size=n // L + 1)
        idx = np.concatenate([np.arange(s, s + L) for s in starts])[:n]
        ub, vb = pseudo_obs(X[idx]), pseudo_obs(Y[idx]); r = fit2(ll_sjc, ub, vb)
        diffs.append(r.x[0] - r.x[1])
    diffs = np.array(diffs)
    B['sjc_bootstrap_tauU_minus_tauL'] = {'point': B['sjc']['tau_U'] - B['sjc']['tau_L'],
        'ci95': [float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))],
        'share_positive': float((diffs > 0).mean()), 'B': 200, 'block': L}
    out['asymmetric'] = B

    # ---------------- out-of-sample: fit on one window, evaluate on the other (ranks within each window)
    def oos(train, test):
        ut, vt = pseudo_obs(X[train]), pseudo_obs(Y[train]); ue, ve = pseudo_obs(X[test]), pseudo_obs(Y[test])
        res = {}
        for f in FAMS:
            p = fit_family(f, ut, vt)['param']
            llf = ll_t if f == 'student_t' else LL[f]
            pts = np.array([llf(p, ue[i:i + 1], ve[i:i + 1]) for i in range(len(ue))])
            res[f] = {'param_train': p, 'll_test': float(pts.sum()), 'pts': pts}
        for name, ll in (('sjc', ll_sjc),):
            r = fit2(ll, ut, vt); pts = np.array([ll(r.x, ue[i:i + 1], ve[i:i + 1]) for i in range(len(ue))])
            res[name] = {'param_train': [float(r.x[0]), float(r.x[1])], 'll_test': float(pts.sum()), 'pts': pts}
        # Diebold-Mariano style comparison of per-observation log scores vs the Gaussian copula (HAC, 10 lags)
        for f in res:
            d = res[f]['pts'] - res['gaussian']['pts']
            if f == 'gaussian': continue
            m = d.mean(); T = len(d); dc = d - m; s = np.sum(dc * dc) / T
            for l in range(1, 11): s += 2 * (1 - l / 11) * np.sum(dc[l:] * dc[:-l]) / T
            res[f]['dm_vs_gaussian_t'] = float(m / np.sqrt(s / T))
        for f in res: res[f].pop('pts')
        return res

    dates = pr.index
    w1 = dates < '2021-01-01'; w2 = ~w1
    out['oos'] = {'train_2018_2020_test_2021_2022': {'n_train': int(w1.sum()), 'n_test': int(w2.sum()), **oos(w1, w2)},
                  'train_2021_2022_test_2018_2020': {'n_train': int(w2.sum()), 'n_test': int(w1.sum()), **oos(w2, w1)}}
    with open(RESULTS / 'copula_extra_checks.json', 'w') as f:
        json.dump(out, f, indent=1)
    A = out['asymmetric']
    print(f"SJC: tau_U={A['sjc']['tau_U']:.3f} tau_L={A['sjc']['tau_L']:.3f}  AIC={A['sjc']['aic']:.2f} | BB7 AIC={A['bb7']['aic']:.2f}")
    print(f"LR test tau_U = tau_L: stat={A['LR_symmetry']['stat']:.2f}  p={A['LR_symmetry']['p_value']:.3f}")
    bb = A['sjc_bootstrap_tauU_minus_tauL']; print(f"Block bootstrap tau_U - tau_L: {bb['point']:.3f}  95% CI [{bb['ci95'][0]:.3f}, {bb['ci95'][1]:.3f}]")
    for split, r in out['oos'].items():
        print(split, {f: round(v['ll_test'], 2) for f, v in r.items() if isinstance(v, dict)})
    print('Saved copula_extra_checks.json')

    return out

copula_extra = _copula_extra_checks()


## Section 9. Option-implied marginals and Monte Carlo pricing

* **9.1** SVI smile per maturity with no-arbitrage penalties; diagnostics (fit RMSE, butterfly g(k), calendar crossings, negative density mass, martingale correction).
* **9.2** Sampler validation: Kendall tau, uniform margins and joint tail frequencies for two antithetic schemes.
* **9.3** Pricing: 72 snapshots × 4 maturities × 5 strikes × 5 copulas × 4 parameter sets × 2 marginal specifications = 57,600 prices, each with its standard error.
* **9.4** Monte Carlo convergence and variance reduction.
* **9.5** Model-free price bounds and dependence-premium stress test.
* **9.6** Sensitivity of the headline cells to the simulation design (Appendix Table A.3).


In [ ]:
# ============================================================
# Cell 9.1 - Smile surfaces (SVI per maturity pillar) and static-arbitrage diagnostics
# For every snapshot and both assets: raw-SVI fit per pillar with penalties for butterfly
# arbitrage (Gatheral-Jacquier g(k) >= 0), Lee wing bound and calendar monotonicity.
# Diagnostics written: fit RMSE (vol points), min g(k), calendar violations, negative
# density mass of the implied marginal, martingale correction E[S_T]/F.
# Output: smile_surfaces.json (SVI parameters per slice), surface_diagnostics.csv   (runtime ~15-25 minutes)
# ============================================================
import time
spx_surf = pd.read_csv(INTERMED / 'SPX_vol_surface_grid.csv')
eur_surf = pd.read_csv(INTERMED / 'EUR_vol_surface_grid.csv')
rates_df = pd.read_csv(INTERMED / 'RATES_lookup.csv', index_col=0, parse_dates=True).ffill()
common_dates = sorted(set(spx_surf.date) & set(eur_surf.date))
print(f'SPX surfaces {spx_surf.date.nunique()} | EUR surfaces {eur_surf.date.nunique()} | common {len(common_dates)}')
print('EUR-only dates (no SPX surface):', sorted(set(eur_surf.date) - set(spx_surf.date)))

surfaces = {}; t0 = time.time()
for i, d in enumerate(common_dates):
    row = rates_df.loc[:d].iloc[-1]
    r, q, rf = float(row['r']), float(row['q']), float(row['r_f'])
    surfaces[d] = dict(r=r, q=q, r_f=rf,
                       spx=SmileSurface(spx_surf[spx_surf.date == d], r, q),
                       eur=SmileSurface(eur_surf[eur_surf.date == d], r, rf))
    if i % 12 == 0: print(f'  {i+1}/{len(common_dates)} {d}  {time.time()-t0:.0f}s', flush=True)
save_surfaces(surfaces, INTERMED / 'smile_surfaces.json')

diag_rows = []
for d, s in surfaces.items():
    for nm, sf, y, kg in [('spx', s['spx'], s['q'], np.linspace(-.6, .4, 101)), ('eur', s['eur'], s['r_f'], np.linspace(-.2, .2, 81))]:
        for T in [0.25, 0.5, 1.0, 2.0]:
            M = ImpliedMarginal(sf, T, np.exp((s['r'] - y) * T))
            diag_rows.append(dict(date=d, asset=nm, T=T, n_pillars=len(sf.slices), max_rmse_vol=max(x[2] for x in sf.slices),
                                  min_g=sf.butterfly_min_g(kg), calendar_violations=sf.calendar_violations(kg),
                                  neg_density_mass=M.neg_mass, mean_ratio=M.mean_ratio, atm_vol=M.atm_vol, source=M.src[0],
                                  source_T=str(M.src[1])))
sdiag = pd.DataFrame(diag_rows); sdiag.to_csv(INTERMED / 'surface_diagnostics.csv', index=False)
print(sdiag.groupby(['asset', 'T']).agg(rmse_med=('max_rmse_vol', 'median'), rmse_max=('max_rmse_vol', 'max'),
      negmass_max=('neg_density_mass', 'max'), mean_ratio_min=('mean_ratio', 'min'), mean_ratio_max=('mean_ratio', 'max')).round(4))
print('Source of the T=2 SPX marginal:', sdiag[(sdiag.asset == 'spx') & (sdiag['T'] == 2.0)].source_T.unique(),
      '-> the SPX surface has no pillar beyond 1.38y; the 2Y SPX smile is extrapolated (flat ATM vol, fixed standardised smile)')


In [ ]:
# ============================================================
# Cell 9.2 - Sampler validation: Kendall tau, uniform margins AND tail concentration
# Compares antithetic reflection of the driving uniforms with reflection of the copula output
# (1-u, 1-v), which turns Clayton and Gumbel draws into their survival copulas. Kendall tau does
# not reveal the difference; joint tail frequencies do.
# Output: sampler_validation.csv
# ============================================================
cal = json.load(open(INTERMED / 'calibration.json'))
P_full = {f: cal['full'][f]['param'] for f in FAMILIES}
rng_v = np.random.default_rng(7); n_v = 500_000; rows = []
for f, p in P_full.items():
    W = rng_v.random((n_v // 2, N_DRIVERS[f]))
    a0, b0 = sample_driven(f, p, W)
    for label, (u, v) in {'driving-uniform reflection': sample_antithetic(f, p, W),
                          'output reflection (1-u,1-v)': (np.r_[a0, 1 - a0], np.r_[b0, 1 - b0])}.items():
        tau = stats.kendalltau(u[:20000], v[:20000])[0]
        for qq in (0.95, 0.99):
            Cqq = float(np.atleast_1d(cdf(f, p, np.array([qq]), np.array([qq])))[0])
            C1q = float(np.atleast_1d(cdf(f, p, np.array([1 - qq]), np.array([1 - qq])))[0])
            rows.append(dict(copula=f, scheme=label, q=qq, tau_emp=tau, tau_theory=implied_tau(f, p),
                             upper_emp=np.mean((u > qq) & (v > qq)) / (1 - qq), upper_theory=(1 - 2 * qq + Cqq) / (1 - qq),
                             lower_emp=np.mean((u < 1 - qq) & (v < 1 - qq)) / (1 - qq), lower_theory=C1q / (1 - qq),
                             ks_uniform_p=stats.kstest(u[:20000], 'uniform').pvalue))
sval = pd.DataFrame(rows).round(4); sval.to_csv(RESULTS / 'sampler_validation.csv', index=False); print(sval.to_string(index=False))


In [ ]:
# ============================================================
# Cell 9.3 - Monte Carlo pricing engine (50/50 SPX / EUR-USD basket call, paid in USD)
#   * Marginals: option-implied (SVI smile -> Breeden-Litzenberger CDF -> inverse CDF), with the
#     flat-smile lognormal benchmark at the ATM-forward vol of the same surface ('lognormal')
#   * Antithetic variates on the DRIVING uniforms (both halves are draws of the intended copula)
#   * Common random numbers: one set of driving uniforms per snapshot shared by all families,
#     parameter sets, maturities, strikes and marginal specifications
#   * SE from N/2 antithetic pair averages; path-wise SE of each copula-minus-Gaussian difference
#   * Strikes K = (K/F_B) * F_B with F_B = 0.5 e^{(r-q)T} + 0.5 e^{(r-r_f)T}
#   * Copula parameter sets: full sample and calm / elevated / stress fits, each applied to every
#     snapshot (separates the volatility channel from the dependence channel)
#   * Seed: 20260 + snapshot index (reproducible per snapshot)
# Output: basket_option_prices.csv (57,600 rows), marginal_diagnostics.csv   (~6-10 minutes)
# ============================================================
import time


MATS = [0.25, 0.5, 1.0, 2.0]
KF = [0.90, 0.95, 1.00, 1.05, 1.10]


def build_marginals(s, T):
    r, q, rf = s['r'], s['q'], s['r_f']
    f1, f2 = np.exp((r - q) * T), np.exp((r - rf) * T)
    m1 = ImpliedMarginal(s['spx'], T, f1); m2 = ImpliedMarginal(s['eur'], T, f2)
    l1 = LognormalMarginal(m1.atm_vol, T, f1); l2 = LognormalMarginal(m2.atm_vol, T, f2)
    diag = dict(atm_spx=m1.atm_vol, atm_eur=m2.atm_vol, negmass_spx=m1.neg_mass, negmass_eur=m2.neg_mass,
                mcorr_spx=1 / m1.mean_ratio, mcorr_eur=1 / m2.mean_ratio,
                src_spx=str(m1.src[0]), src_eur=str(m2.src[0]), f1=f1, f2=f2)
    return {'smile': (m1, m2), 'lognormal': (l1, l2)}, diag


def price_snapshot(d, s, param_sets, N, seed, specs=('smile', 'lognormal'), mats=MATS, kf=KF):
    rng = np.random.default_rng(seed)
    n2 = N // 2
    W = rng.random((n2, 4))                       # common driving uniforms (CRN)
    draws = {}
    for ps, pars in param_sets.items():
        for fam in PRICING_FAMILIES:
            draws[(ps, fam)] = sample_antithetic(fam, pars[fam], W[:, :N_DRIVERS[fam]])
    rows, diags = [], []
    for T in mats:
        margs, diag = build_marginals(s, T)
        diag.update(date=d, maturity=T); diags.append(diag)
        r = s['r']; disc = np.exp(-r * T)
        FB = 0.5 * diag['f1'] + 0.5 * diag['f2']
        for spec in specs:
            M1, M2 = margs[spec]
            for ps in param_sets:
                Y = {}
                for fam in PRICING_FAMILIES:
                    u, v = draws[(ps, fam)]
                    B = 0.5 * M1.ppf(u) + 0.5 * M2.ppf(v)
                    for k in kf:
                        pay = disc * np.maximum(B - k * FB, 0.0)
                        Y[(fam, k)] = 0.5 * (pay[:n2] + pay[n2:])        # antithetic pair averages
                for k in kf:
                    yg = Y[('gaussian', k)]; pg = yg.mean()
                    for fam in PRICING_FAMILIES:
                        y = Y[(fam, k)]; p = y.mean()
                        se = y.std(ddof=1) / np.sqrt(n2)
                        if fam == 'gaussian':
                            dpct, se_d = 0.0, 0.0
                        else:
                            ratio = p / pg
                            dpct = 100 * (ratio - 1)
                            se_d = 100 * (y - ratio * yg).std(ddof=1) / np.sqrt(n2) / pg
                        rows.append(dict(date=d, maturity=T, strike_KF=k, spec=spec, params=ps, copula=fam,
                                         price=p, se=se, diff_pct=dpct, se_diff_pct=se_d,
                                         K_abs=k * FB, FB=FB, r=r, q=s['q'], r_f=s['r_f']))
    return rows, diags


def param_sets_from(cal, sets=('full', 'calm_prev', 'elevated_prev', 'stress_prev')):
    out = {}
    for sname in sets:
        out[sname.replace('_prev', '')] = {f: cal[sname][f]['param'] for f in PRICING_FAMILIES}
    return out



surfaces = load_surfaces(INTERMED / 'smile_surfaces.json')
cal = json.load(open(INTERMED / 'calibration.json'))
PS = param_sets_from(cal)
for k, v in PS.items(): print(k, {f: np.round(p, 4) for f, p in v.items()})
allrows, alldiag = [], []; t0 = time.time()
for i, d in enumerate(sorted(surfaces)):
    rws, dg = price_snapshot(d, surfaces[d], PS, 100_000, seed=20260 + i)
    allrows += rws; alldiag += dg
    if i % 12 == 0: print(f'  {i+1}/{len(surfaces)} {d}  {time.time()-t0:.0f}s', flush=True)
bp_new = pd.DataFrame(allrows); bp_new.to_csv(INTERMED / 'basket_option_prices.csv', index=False)
bp_new.to_csv(RESULTS / 'basket_option_prices.csv', index=False)
pd.DataFrame(alldiag).to_csv(INTERMED / 'marginal_diagnostics.csv', index=False)
print(f'Done: {len(bp_new):,} rows = 72 snapshots x 4 maturities x 5 strikes x 5 families x 4 parameter sets x 2 marginal specs')
chk = bp_new[(bp_new.spec == 'smile') & (bp_new.params == 'full') & (bp_new.maturity == 1.0) & (bp_new.strike_KF == 1.0)]
print(chk.groupby('copula').price.mean().round(6))


In [ ]:
# ============================================================
# Cell 9.4 - Monte Carlo convergence experiment
# Snapshot 2020-07-31, T = 1y, K/F_B in {1.00, 1.10}, Gaussian and Gumbel, option-implied marginals.
# 30 independent seeds per N in {1k, ..., 200k}; antithetic vs plain MC at the same N.
# Output: convergence_runs.csv, convergence_summary.csv, antithetic_variance_reduction.csv
# ============================================================


surf = load_surfaces(INTERMED / 'smile_surfaces.json'); cal = json.load(open(INTERMED / 'calibration.json'))
d, T = '2020-07-31', 1.0
s = surf[d]; r, q, rf = s['r'], s['q'], s['r_f']
f1, f2 = np.exp((r - q) * T), np.exp((r - rf) * T); FB = 0.5 * f1 + 0.5 * f2; disc = np.exp(-r * T)
M1 = ImpliedMarginal(s['spx'], T, f1); M2 = ImpliedMarginal(s['eur'], T, f2)
P = {f: cal['full'][f]['param'] for f in ['gaussian', 'gumbel']}
rows = []
for N in [1_000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000, 200_000]:
    for rep in range(30):
        rng = np.random.default_rng(10_000 + 97 * rep + N)
        W = rng.random((N // 2, 4))
        Wp = rng.random((N, 4))                                   # plain MC with the same budget
        for fam, p in P.items():
            u, v = sample_antithetic(fam, p, W[:, :N_DRIVERS[fam]])
            B = 0.5 * M1.ppf(u) + 0.5 * M2.ppf(v)
            up, vp = sample_driven(fam, p, Wp[:, :N_DRIVERS[fam]])
            Bp = 0.5 * M1.ppf(up) + 0.5 * M2.ppf(vp)
            for k in (1.00, 1.10):
                pay = disc * np.maximum(B - k * FB, 0); Y = 0.5 * (pay[:N // 2] + pay[N // 2:])
                payp = disc * np.maximum(Bp - k * FB, 0)
                rows.append(dict(N=N, rep=rep, copula=fam, KF=k, price=Y.mean(), se=Y.std(ddof=1) / np.sqrt(N // 2),
                                 price_plain=payp.mean(), se_plain=payp.std(ddof=1) / np.sqrt(N)))
df = pd.DataFrame(rows); df.to_csv(RESULTS / 'convergence_runs.csv', index=False)
g = df.groupby(['copula', 'KF', 'N']).agg(mean_price=('price', 'mean'), sd_across_seeds=('price', 'std'),
                                           mean_reported_se=('se', 'mean'), sd_plain=('price_plain', 'std')).reset_index()
g['var_reduction_factor'] = (g.sd_plain / g.sd_across_seeds) ** 2
g.to_csv(RESULTS / 'convergence_summary.csv', index=False)
print(g.to_string(index=False))

vr = df.assign(v_anti=df.se**2 * df.N, v_plain=df.se_plain**2 * df.N).groupby(['copula', 'KF'])[['v_anti', 'v_plain']].mean()
vr['variance_reduction_factor'] = vr.v_plain / vr.v_anti
print(vr.round(4)); vr.to_csv(RESULTS / 'antithetic_variance_reduction.csv')


In [ ]:
# ============================================================
# Cell 9.5 - Measure check: how much could prices move if risk-neutral dependence differs from historical?
#   1. Model-free bounds: for a two-asset basket call the countermonotonic and comonotonic copulas give the
#      lowest and highest price compatible with the two option-implied marginals (Tchen, 1980)
#   2. Dependence-premium stress: every family re-parameterised to Kendall tau 0.10 below / above its
#      full-sample estimate (Student-t nu kept), one-year grid repriced with the SAME random numbers
# Output: qmeasure_prices.csv, table_qmeasure.csv, qmeasure_params.json   (~6 minutes)
# ============================================================
from scipy.optimize import brentq
# Self-sufficient: works after a runtime restart once Cell 8.0 has run (no need to rerun the pricing of Cell 9.3)
if 'surfaces' not in globals():
    surfaces = load_surfaces(INTERMED / 'smile_surfaces.json')
if 'cal' not in globals():
    cal = json.load(open(INTERMED / 'calibration.json'))
if 'KF' not in globals():
    KF = [0.90, 0.95, 1.00, 1.05, 1.10]
if 'build_marginals' not in globals():
    def build_marginals(s, T):
        r, q, rf = s['r'], s['q'], s['r_f']
        f1, f2 = np.exp((r - q) * T), np.exp((r - rf) * T)
        m1 = ImpliedMarginal(s['spx'], T, f1); m2 = ImpliedMarginal(s['eur'], T, f2)
        l1 = LognormalMarginal(m1.atm_vol, T, f1); l2 = LognormalMarginal(m2.atm_vol, T, f2)
        diag = dict(atm_spx=m1.atm_vol, atm_eur=m2.atm_vol, negmass_spx=m1.neg_mass, negmass_eur=m2.neg_mass,
                    mcorr_spx=1 / m1.mean_ratio, mcorr_eur=1 / m2.mean_ratio,
                    src_spx=str(m1.src[0]), src_eur=str(m2.src[0]), f1=f1, f2=f2)
        return {'smile': (m1, m2), 'lognormal': (l1, l2)}, diag

def _measure_check():
    T, N = 1.0, 100_000
    full = {f: cal['full'][f]['param'] for f in PRICING_FAMILIES}
    tauP = {f: float(implied_tau(f, full[f])) for f in PRICING_FAMILIES}
    def param_for_tau(f, tau):
        if f == 'gaussian': return float(np.sin(np.pi * tau / 2))
        if f == 'student_t': return [float(np.sin(np.pi * tau / 2)), full['student_t'][1]]
        if f == 'clayton': return 2 * tau / (1 - tau)
        if f == 'gumbel': return 1 / (1 - tau)
        if f == 'frank': return brentq(lambda t: implied_tau('frank', t) - tau, 1e-6, 60)
    SHIFT = {'down': -0.10, 'base': 0.0, 'up': +0.10}
    PSq = {k: {f: (full[f] if dv == 0 else param_for_tau(f, tauP[f] + dv)) for f in PRICING_FAMILIES} for k, dv in SHIFT.items()}
    vx = pd.read_csv(INTERMED / 'VIX_daily.csv'); vx = vx.rename(columns={vx.columns[0]: 'Date'})
    vx['Date'] = pd.to_datetime(vx['Date']).dt.strftime('%Y-%m-%d'); vmap = dict(zip(vx.Date, vx.Close))
    reg = lambda x: 'calm' if x < 20 else ('elevated' if x < 30 else 'stress')
    rows = []
    for i, d in enumerate(sorted(surfaces)):
        s = surfaces[d]; rng = np.random.default_rng(20260 + i); n2 = N // 2
        W = rng.random((n2, 4))                                    # identical to Cell 9.3
        margs, diag = build_marginals(s, T); M1, M2 = margs['smile']
        disc = np.exp(-s['r'] * T); FB = 0.5 * diag['f1'] + 0.5 * diag['f2']
        U = np.r_[W[:, 0], 1 - W[:, 0]]
        B_co = 0.5 * M1.ppf(U) + 0.5 * M2.ppf(U); B_ct = 0.5 * M1.ppf(U) + 0.5 * M2.ppf(1 - U)
        for k in KF:
            K = k * FB
            rec = dict(date=d, KF=k, regime=reg(vmap[d]), upper=disc * np.maximum(B_co - K, 0).mean(),
                       lower=disc * np.maximum(B_ct - K, 0).mean())
            for ps, pars in PSq.items():
                for f in PRICING_FAMILIES:
                    u, v = sample_antithetic(f, pars[f], W[:, :N_DRIVERS[f]])
                    rec[f'{ps}_{f}'] = disc * np.maximum(0.5 * M1.ppf(u) + 0.5 * M2.ppf(v) - K, 0).mean()
            rows.append(rec)
        if i % 12 == 0: print(f'  {i+1}/{len(surfaces)} {d}', flush=True)
    df = pd.DataFrame(rows); df.to_csv(RESULTS / 'qmeasure_prices.csv', index=False)
    out = []
    for (k, g), x in df.groupby(['KF', 'regime']):
        pg = x['base_gaussian']; fam = lambda ps: x[[f'{ps}_{f}' for f in PRICING_FAMILIES]]
        r = dict(KF=k, regime=g, n=len(x), gauss=pg.mean(), lower_pct=(100 * (x.lower / pg - 1)).mean(),
                 upper_pct=(100 * (x.upper / pg - 1)).mean())
        for ps in SHIFT:
            P = fam(ps).values
            r[f'spread_{ps}'] = (100 * (P.max(axis=1) - P.min(axis=1)) / pg).mean()
            r[f'gumbel_minus_frank_{ps}'] = (100 * (x[f'{ps}_gumbel'] - x[f'{ps}_frank']) / pg).mean()
            r[f'gauss_shift_{ps}'] = (100 * (x[f'{ps}_gaussian'] / pg - 1)).mean()
        r['share_of_bounds_base'] = ((fam('base').max(axis=1) - fam('base').min(axis=1)) / (x.upper - x.lower)).mean()
        out.append(r)
    tab = pd.DataFrame(out)
    tab['_o'] = tab.regime.map({'calm': 0, 'elevated': 1, 'stress': 2}); tab = tab.sort_values(['KF', '_o']).drop(columns='_o')
    tab.to_csv(RESULTS / 'table_qmeasure.csv', index=False)
    json.dump({'tau_P': tauP, 'params': PSq}, open(RESULTS / 'qmeasure_params.json', 'w'), indent=1, default=float)
    return tab

tab_q = _measure_check()
print(tab_q[tab_q.KF.isin([1.0, 1.1])].round(2).to_string(index=False))


In [ ]:
# ============================================================
# Cell 9.6 - Sensitivity of the headline cells to the simulation design (Appendix Table A.3)
# One-year K/F_B = 1.10 differences from the Gaussian copula, by regime, as the design is built up:
#   (0) antithetic pairs formed by reflecting the copula output, nearest-node ATM vol, lognormal marginals,
#       strike relative to the initial basket value, previous month's Euribor average
#   (1) antithetic pairs formed on the driving uniforms, common random numbers across copulas
#   (2) strike relative to the basket forward F_B
#   (3) as-of Euribor (Cell 2.2b) and ATM vol interpolated in total variance
#   (4) smile-implied marginals = main specification of Cell 9.3 (here with seed 777 + snapshot index)
# Output: table_A3_design_sensitivity.csv   (~1 minute)
# ============================================================
surf = load_surfaces(INTERMED / 'smile_surfaces.json'); cal = json.load(open(INTERMED / 'calibration.json'))
spx_g = pd.read_csv(INTERMED / 'SPX_vol_surface_grid.csv'); eur_g = pd.read_csv(INTERMED / 'EUR_vol_surface_grid.csv')
rl = pd.read_csv(INTERMED / 'RATES_lookup.csv', index_col=0, parse_dates=True)
own_month = rl['r_f'].groupby(rl.index.to_period('M')).last()      # complete own-month average at each month end
rf_prev = lambda d: float(own_month.get(pd.Period(d, 'M') - 1, own_month.iloc[0]))
vx = pd.read_csv(INTERMED / 'VIX_daily.csv'); vx = vx.rename(columns={vx.columns[0]: 'Date'})
vx['Date'] = pd.to_datetime(vx['Date']).dt.strftime('%Y-%m-%d'); vmap = dict(zip(vx.Date, vx.Close))
reg = lambda v: 'calm' if v < 20 else ('elevated' if v < 30 else 'stress')
FAM = ['gaussian', 'frank', 'clayton', 'gumbel']; PAR = {f: cal['full'][f]['param'] for f in FAM}


def atm_nearest(g, d, T):
    a = g[(g.date == d) & (g.moneyness >= .99) & (g.moneyness <= 1.01)]
    return float(a.loc[(a.ttm_years - T).abs().idxmin(), 'impl_vol'])


rows = []; N, T = 100_000, 1.0; n2 = N // 2
for i, d in enumerate(sorted(surf)):
    s = surf[d]; r, q, rf = s['r'], s['q'], s['r_f']; rfo = rf_prev(d)
    W = np.random.default_rng(777 + i).random((n2, 4))
    d_new = {f: sample_antithetic(f, PAR[f], W[:, :N_DRIVERS[f]]) for f in FAM}
    d_old = {}
    for f in FAM:
        a, b = sample_driven(f, PAR[f], W[:, :N_DRIVERS[f]]); d_old[f] = (np.r_[a, 1 - a], np.r_[b, 1 - b])
    so, eo = atm_nearest(spx_g, d, T), atm_nearest(eur_g, d, T)
    m1 = ImpliedMarginal(s['spx'], T, np.exp((r - q) * T)); m2 = ImpliedMarginal(s['eur'], T, np.exp((r - rf) * T))
    LN = lambda u, sig, mu: np.exp((mu - .5 * sig * sig) * T + sig * np.sqrt(T) * norm.ppf(u))
    FBo = .5 * np.exp((r - q) * T) + .5 * np.exp((r - rfo) * T); FBn = .5 * np.exp((r - q) * T) + .5 * np.exp((r - rf) * T)
    steps = {0: (d_old, lambda u, v: .5 * LN(u, so, r - q) + .5 * LN(v, eo, r - rfo), 1.0),
             1: (d_new, lambda u, v: .5 * LN(u, so, r - q) + .5 * LN(v, eo, r - rfo), 1.0),
             2: (d_new, lambda u, v: .5 * LN(u, so, r - q) + .5 * LN(v, eo, r - rfo), FBo),
             3: (d_new, lambda u, v: .5 * LN(u, m1.atm_vol, r - q) + .5 * LN(v, m2.atm_vol, r - rf), FBn),
             4: (d_new, lambda u, v: .5 * m1.ppf(u) + .5 * m2.ppf(v), FBn)}
    for st, (dr, basket, ref) in steps.items():
        pr = {f: np.exp(-r * T) * np.maximum(basket(*dr[f]) - 1.10 * ref, 0).mean() for f in FAM}
        for f in FAM[1:]:
            rows.append(dict(step=st, date=d, regime=reg(vmap[d]), copula=f, diff_pct=100 * (pr[f] / pr['gaussian'] - 1)))
tab_a3 = pd.DataFrame(rows).groupby(['regime', 'copula', 'step']).diff_pct.mean().unstack('step').round(2)
tab_a3.to_csv(RESULTS / 'table_A3_design_sensitivity.csv'); print(tab_a3.to_string())


## Section 10. Tables, figures and results

Builds every table and figure reported in the thesis, and `results_full.json` with all the reported numbers, from the intermediate files written by Sections 1-9 to `thesis_processed/`.

If `thesis_processed/` is already populated, this section can be run on its own: select Cell 10.0 and use **Runtime → Run cell and below** (about 2 minutes). Cell 10.0 checks that every required intermediate file is present and lists any that are missing.

Outputs are written to `outputs/`:
- `results_full.json`: every number reported in the thesis
- `tables/`: Chapter 5 tables, standardised-moneyness inputs, snapshot inventory and `headline.json`
- `figures/`: Figures 4.1-4.5 and 5.1-5.10


In [ ]:
# ============================================================
# Cell 10.0 - Drive mount and intermediate-files check
# ============================================================
# (a) Mounts Drive if the project folder lives there and Drive is not mounted yet
# (b) Checks that every intermediate file needed by Section 10 exists
# (c) Lists the missing files, if any

import os
from pathlib import Path

# --- (a) Project folder (same rule as Cell 1.1) --------------------------
ROOT     = Path(os.environ.get('THESIS_ROOT', '/content/drive/MyDrive/Colab Notebooks'))
if str(ROOT).startswith('/content/drive') and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

# --- (b) Define paths and verify intermediates ---------------------------
INTERMED = ROOT / 'thesis_processed'
RESULTS  = ROOT / 'results'
OUTPUTS   = ROOT / 'outputs'
TABLES  = OUTPUTS / 'tables'
FIGURES  = OUTPUTS / 'figures'

required = {
    'basket_option_prices'      : INTERMED / 'basket_option_prices.csv',
    'paired_returns'            : INTERMED / 'PAIRED_returns_copula.csv',
    'vix_daily'                 : INTERMED / 'VIX_daily.csv',
    'calibration'               : INTERMED / 'calibration.json',
    'copula_robustness'         : INTERMED / 'copula_robustness.json',
    'smile_surfaces'            : INTERMED / 'smile_surfaces.json',
    'marginal_diagnostics'      : INTERMED / 'marginal_diagnostics.csv',
    'spx_vol_grid'              : INTERMED / 'SPX_vol_surface_grid.csv',
    'eur_vol_grid'              : INTERMED / 'EUR_vol_surface_grid.csv',
    'spx_daily'                 : INTERMED / 'SPX_daily_prices.csv',
    'eur_daily'                 : INTERMED / 'EURUSD_daily_spot.csv',
    'rates'                     : INTERMED / 'RATES_lookup.csv',
    'div_yield'                 : INTERMED / 'SPX_implied_dividend_yield.csv',
}

print()
print('=' * 64)
print(' INTERMEDIATE-FILES AUDIT')
print('=' * 64)
print(f' Looking in: {INTERMED}')
print()

missing = []
found   = []
for key, path in required.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        found.append(key)
        print(f'  [OK]   {path.name:48s} {size_mb:>8.2f} MB')
    else:
        missing.append(key)
        print(f'  [MISS] {path.name}')

print()
if missing:
    print('=' * 64)
    print(f' MISSING {len(missing)} INTERMEDIATE FILE(S)')
    print('=' * 64)
    print(' Run Sections 1-9 first to create them in')
    print(f' {INTERMED}')
    print()
    raise FileNotFoundError(f'Missing intermediates: {missing}')
else:
    print('=' * 64)
    print(' All intermediate files found.')
    print('=' * 64)
    for d in (OUTPUTS, TABLES, FIGURES):
        d.mkdir(parents=True, exist_ok=True)
    print(f' Outputs will be written to: {OUTPUTS}')


In [ ]:
# ============================================================
# Cell 10.1 - Load intermediate files
# ============================================================
import os, json, zipfile, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import rankdata, norm
from scipy.integrate import quad
from datetime import datetime, timezone

ROOT     = Path(os.environ.get('THESIS_ROOT', '/content/drive/MyDrive/Colab Notebooks'))
INTERMED = ROOT / 'thesis_processed'
RESULTS  = ROOT / 'results'
OUTPUTS   = ROOT / 'outputs'
TABLES  = OUTPUTS / 'tables'
FIGURES  = OUTPUTS / 'figures'
for d in (OUTPUTS, TABLES, FIGURES, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

bp        = pd.read_csv(INTERMED / 'basket_option_prices.csv')
pr        = pd.read_csv(INTERMED / 'PAIRED_returns_copula.csv')
pr        = pr.rename(columns={pr.columns[0]: 'Date'}); pr['Date'] = pd.to_datetime(pr['Date'])
vix       = pd.read_csv(INTERMED / 'VIX_daily.csv'); vix = vix.rename(columns={vix.columns[0]: 'Date'})
vix['Date'] = pd.to_datetime(vix['Date'])
pr['VIX'] = pr['Date'].map(dict(zip(vix.Date, vix.Close)))
spx_grid  = pd.read_csv(INTERMED / 'SPX_vol_surface_grid.csv')
eur_grid  = pd.read_csv(INTERMED / 'EUR_vol_surface_grid.csv')
spx_daily = pd.read_csv(INTERMED / 'SPX_daily_prices.csv'); spx_daily = spx_daily.rename(columns={spx_daily.columns[0]: 'Date'}); spx_daily['Date'] = pd.to_datetime(spx_daily['Date'])
eur_daily = pd.read_csv(INTERMED / 'EURUSD_daily_spot.csv'); eur_daily = eur_daily.rename(columns={eur_daily.columns[0]: 'Date'}); eur_daily['Date'] = pd.to_datetime(eur_daily['Date'])
rates     = pd.read_csv(INTERMED / 'RATES_lookup.csv', index_col=0, parse_dates=True)
div_yield = pd.read_csv(INTERMED / 'SPX_implied_dividend_yield.csv', parse_dates=['date'])
calib     = json.load(open(INTERMED / 'calibration.json'))
robust    = json.load(open(INTERMED / 'copula_robustness.json'))
print(f'basket_option_prices : {len(bp):,} rows, {bp.date.nunique()} snapshots')
print(f'paired_returns       : {len(pr):,} rows; VIX missing on {pr.VIX.isna().sum()} return dates')
print(f'SPX / EUR surfaces   : {spx_grid.date.nunique()} / {eur_grid.date.nunique()} dates')


In [ ]:
# ============================================================
# Cell 10.2 - Full-sample paired-returns statistics (VIX = Cboe close)
# ============================================================
def pseudo(x):
    return rankdata(x) / (len(x) + 1)

n_paired = len(pr)
pearson_full  = float(pr['SPX_LogReturn'].corr(pr['EUR_LogReturn']))
spearman_full = float(pr['SPX_LogReturn'].corr(pr['EUR_LogReturn'], method='spearman'))
kendall_full  = float(pr['SPX_LogReturn'].corr(pr['EUR_LogReturn'], method='kendall'))

paired_stats = {
    'n_observations'     : int(n_paired),
    'date_min'           : str(pr['Date'].min().date()),
    'date_max'           : str(pr['Date'].max().date()),
    'pearson_correlation': round(pearson_full, 4),
    'spearman_correlation': round(spearman_full, 4),
    'kendall_tau'        : round(kendall_full, 4),
    'spx_log_mean'       : round(float(pr['SPX_LogReturn'].mean()), 6),
    'spx_log_std'        : round(float(pr['SPX_LogReturn'].std()), 6),
    'eur_log_mean'       : round(float(pr['EUR_LogReturn'].mean()), 6),
    'eur_log_std'        : round(float(pr['EUR_LogReturn'].std()), 6),
    'n_missing_vix_in_paired_sample': int(pr['VIX'].isna().sum()),
}
for k, v in paired_stats.items():
    print(f'  {k:35s} = {v}')


In [ ]:
# ============================================================
# Cell 10.3 - Chapter 5 tables
# Main specification: option-implied (smile) marginals, full-sample copula ('smile','full').
# Regime of a snapshot = Cboe VIX close on the snapshot date (calm < 20 <= elevated < 30 <= stress);
# percentile (tercile) thresholds as robustness. Table 5.4 is reported by maturity (1Y in text).
# Output (TABLES): table_5_2*, table_5_3*, table_5_4*, table_5_5_decomposition*, table_zmoneyness_bins,
#                   table_case_studies, table_smile_vs_lognormal, table_A1_inventory, headline.json
# ============================================================
import statsmodels.formula.api as smf


P = pd.read_csv(INTERMED / 'basket_option_prices.csv')
vix = pd.read_csv(INTERMED / 'VIX_daily.csv', index_col=0, parse_dates=True)['Close']
dates = sorted(P.date.unique())
vsnap = {d: float(vix.asof(pd.Timestamp(d))) for d in dates}
reg = lambda v: 'calm' if v < 20 else ('elevated' if v < 30 else 'stress')
P['vix'] = P.date.map(vsnap); P['regime'] = P.vix.map(reg)
# percentile-threshold robustness: terciles of the daily VIX distribution over the sample window
vw = vix['2018-01-01':'2022-12-31']; t1, t2 = np.percentile(vw, [100 / 3, 200 / 3])
P['regime_pct'] = P.vix.map(lambda v: 'calm' if v < t1 else ('elevated' if v < t2 else 'stress'))
P['z_diff'] = np.where(P.se_diff_pct > 0, P.diff_pct / P.se_diff_pct.replace(0, np.nan), np.nan)
FAMS = ['frank', 'clayton', 'gumbel', 'student_t']
H = {'vix_terciles': [float(t1), float(t2)]}
inv = pd.DataFrame({'date': dates, 'vix': [vsnap[d] for d in dates]}); inv['regime'] = inv.vix.map(reg)
inv['regime_pct'] = inv.vix.map(lambda v: 'calm' if v < t1 else ('elevated' if v < t2 else 'stress'))
inv.to_csv(TABLES / 'table_A1_inventory.csv', index=False)
H['regime_counts'] = inv.regime.value_counts().to_dict(); H['regime_pct_counts'] = inv.regime_pct.value_counts().to_dict()

def sub(spec='smile', params='full'):
    return P[(P.spec == spec) & (P.params == params)]

# ---------------------------------------------------------------- Table 5.2: ATM by maturity
def table_atm(spec, params):
    s = sub(spec, params); s = s[s.strike_KF == 1.0]
    rows = []
    for T, g in s.groupby('maturity'):
        row = {'maturity': T, 'gaussian_price': g[g.copula == 'gaussian'].price.mean()}
        for f in FAMS:
            gf = g[g.copula == f]
            row[f + '_mean'] = gf.diff_pct.mean(); row[f + '_min'] = gf.diff_pct.min(); row[f + '_max'] = gf.diff_pct.max()
            row[f + '_share_sig'] = (gf.z_diff.abs() >= 2).mean()
        rows.append(row)
    return pd.DataFrame(rows)
t52 = table_atm('smile', 'full'); t52.to_csv(TABLES / 'table_5_2_atm.csv', index=False)
t52_ln = table_atm('lognormal', 'full'); t52_ln.to_csv(TABLES / 'table_5_2b_atm_lognormal.csv', index=False)

# ---------------------------------------------------------------- Table 5.3: full grid distribution
def table_grid(spec, params):
    s = sub(spec, params); rows = []
    for f in FAMS:
        g = s[s.copula == f].diff_pct
        rows.append(dict(copula=f, N=len(g), mean=g.mean(), median=g.median(), std=g.std(), min=g.min(), max=g.max(),
                         p5=g.quantile(.05), p95=g.quantile(.95),
                         share_sig=(s[s.copula == f].z_diff.abs() >= 2).mean()))
    return pd.DataFrame(rows)
t53 = table_grid('smile', 'full'); t53.to_csv(TABLES / 'table_5_3_grid.csv', index=False)

# ---------------------------------------------------------------- Table 5.4: regime x strike (by maturity)
def table_rs(spec, params, regcol='regime', matched=False):
    s = P[P.spec == spec]
    if matched:   # each snapshot priced with the copula fitted on its own regime
        s = s[s.params == s[regcol]]
    else:
        s = s[s.params == params]
    rows = []
    for (T, k, r), g in s.groupby(['maturity', 'strike_KF', regcol]):
        row = dict(maturity=T, KF=k, regime=r, n=g.date.nunique())
        for f in FAMS:
            x = g[g.copula == f]
            row[f + '_mean'] = x.diff_pct.mean(); row[f + '_sd'] = x.diff_pct.std()
            row[f + '_share_sig'] = (x.z_diff.abs() >= 2).mean()
        row['gumbel_minus_frank'] = row['gumbel_mean'] - row['frank_mean']
        rows.append(row)
    return pd.DataFrame(rows)
t54 = table_rs('smile', 'full'); t54.to_csv(TABLES / 'table_5_4_regime_strike_full.csv', index=False)
t54m = table_rs('smile', None, matched=True); t54m.to_csv(TABLES / 'table_5_4b_regime_strike_matched.csv', index=False)
t54p = table_rs('smile', 'full', regcol='regime_pct'); t54p.to_csv(TABLES / 'table_5_4c_regime_pct.csv', index=False)
t54ln = table_rs('lognormal', 'full'); t54ln.to_csv(TABLES / 'table_5_4d_regime_strike_lognormal.csv', index=False)

# ---------------------------------------------------------------- Decomposition: marginals regime x copula regime
dec = []
s = P[(P.spec == 'smile') & (P.maturity == 1.0) & (P.strike_KF == 1.10)]
for rm in ['calm', 'elevated', 'stress']:
    for pc in ['calm', 'elevated', 'stress', 'full']:
        g = s[(s.regime == rm) & (s.params == pc)]
        row = dict(snapshot_regime=rm, copula_params=pc, n=g.date.nunique())
        for f in FAMS: row[f] = g[g.copula == f].diff_pct.mean()
        row['gumbel_minus_frank'] = row['gumbel'] - row['frank']
        dec.append(row)
dec = pd.DataFrame(dec); dec.to_csv(TABLES / 'table_5_5_decomposition_1Y_110.csv', index=False)

# ---------------------------------------------------------------- Standardised moneyness and regression
s = sub('smile', 'full').copy()
g0 = s[s.copula == 'gaussian'][['date', 'maturity', 'strike_KF', 'price']].rename(columns={'price': 'pg'})
diag = pd.read_csv(INTERMED / 'marginal_diagnostics.csv')
s = s.merge(diag[['date', 'maturity', 'atm_spx', 'atm_eur', 'f1', 'f2']], on=['date', 'maturity'])
cal = json.load(open(INTERMED / 'calibration.json')); rho = cal['full']['gaussian']['param']
w1 = 0.5 * s.f1 / (0.5 * s.f1 + 0.5 * s.f2); w2 = 1 - w1
s['sigma_B'] = np.sqrt((w1 * s.atm_spx) ** 2 + (w2 * s.atm_eur) ** 2 + 2 * w1 * w2 * rho * s.atm_spx * s.atm_eur)
s['z_moneyness'] = np.log(s.strike_KF) / (s.sigma_B * np.sqrt(s.maturity))
s.to_csv(TABLES / 'grid_with_zmoneyness.csv', index=False)
gu = s[(s.copula == 'gumbel')].copy()
gu['stress'] = (gu.regime == 'stress').astype(int); gu['elev'] = (gu.regime == 'elevated').astype(int)
# Regime-only and regime + standardised moneyness on the SAME cells (all K/F >= 1.00), so the R2 are comparable
_otm = gu[gu.strike_KF >= 1.0]
m1 = smf.ols('diff_pct ~ stress + elev', data=_otm).fit(cov_type='cluster', cov_kwds={'groups': _otm.date})
m2 = smf.ols('diff_pct ~ stress + elev + z_moneyness + I(z_moneyness**2)', data=gu[gu.strike_KF >= 1.0]).fit(cov_type='cluster', cov_kwds={'groups': gu[gu.strike_KF >= 1.0].date})
H['reg_gumbel_regime_only'] = {k: [float(m1.params[k]), float(m1.bse[k])] for k in m1.params.index}
H['reg_gumbel_with_z'] = {k: [float(m2.params[k]), float(m2.bse[k])] for k in m2.params.index}
H['reg_gumbel_with_z_r2'] = float(m2.rsquared); H['reg_gumbel_regime_only_r2'] = float(m1.rsquared)
# Robustness: the same pair of regressions on the one-year cells only
_otm1 = gu[(gu.strike_KF >= 1.0) & (gu.maturity == 1.0)]
m3 = smf.ols('diff_pct ~ stress + elev', data=_otm1).fit(cov_type='cluster', cov_kwds={'groups': _otm1.date})
m4 = smf.ols('diff_pct ~ stress + elev + z_moneyness + I(z_moneyness**2)', data=_otm1).fit(cov_type='cluster', cov_kwds={'groups': _otm1.date})
H['reg_gumbel_regime_only_1Y'] = {k: [float(m3.params[k]), float(m3.bse[k])] for k in m3.params.index}
H['reg_gumbel_with_z_1Y'] = {k: [float(m4.params[k]), float(m4.bse[k])] for k in m4.params.index}
H['reg_gumbel_regime_only_1Y_r2'] = float(m3.rsquared); H['reg_gumbel_with_z_1Y_r2'] = float(m4.rsquared)
H['reg_n'] = {'all_maturities': int(len(_otm)), 'one_year': int(len(_otm1))}
print(m1.summary().tables[1]); print(m2.summary().tables[1])
# z-moneyness bins
bins = [-1, 0, 0.25, 0.5, 0.75, 1.0, 1.5, 3.0]
s['zbin'] = pd.cut(s.z_moneyness, bins)
zb = s[s.copula.isin(FAMS)].groupby(['zbin', 'copula'], observed=True).diff_pct.mean().unstack()
zb['n'] = s[s.copula == 'gumbel'].groupby('zbin', observed=True).size()
zb.to_csv(TABLES / 'table_zmoneyness_bins.csv'); print(zb)
H['zmoneyness_by_regime_K110_1Y'] = s[(s.copula == 'gumbel') & (s.strike_KF == 1.10) & (s.maturity == 1.0)].groupby('regime').z_moneyness.mean().to_dict()

# ---------------------------------------------------------------- MC precision
m = sub('smile', 'full'); rel = (m.se / m.price).replace([np.inf], np.nan).dropna() * 100
H['mc'] = dict(abs_se_median=float(m.se.median()), abs_se_p95=float(m.se.quantile(.95)), abs_se_max=float(m.se.max()),
               rel_se_median=float(rel.median()), rel_se_p95=float(rel.quantile(.95)), rel_se_max=float(rel.max()),
               share_rel_below_1=float((rel < 1).mean()), share_rel_below_2=float((rel < 2).mean()),
               diff_se_median=float(m[m.copula != 'gaussian'].se_diff_pct.median()),
               diff_se_p95=float(m[m.copula != 'gaussian'].se_diff_pct.quantile(.95)),
               share_diff_sig=float((m[m.copula != 'gaussian'].z_diff.abs() >= 2).mean()))
# ---------------------------------------------------------------- Case studies (1Y)
cases = {'calm': '2019-06-28', 'stress': '2020-10-30', 'elevated': '2021-09-30'}
cs = []
for r, d in cases.items():
    g = sub('smile', 'full'); g = g[(g.date == d) & (g.maturity == 1.0)]
    for k in sorted(g.strike_KF.unique()):
        x = g[g.strike_KF == k]; row = dict(case=r, date=d, vix=vsnap[d], regime_cboe=reg(vsnap[d]), KF=k,
                                             gaussian_price=float(x[x.copula == 'gaussian'].price.iloc[0]))
        for f in FAMS:
            y = x[x.copula == f].iloc[0]; row[f] = y.diff_pct; row[f + '_se'] = y.se_diff_pct
        cs.append(row)
cs = pd.DataFrame(cs); cs.to_csv(TABLES / 'table_case_studies.csv', index=False)
diag[diag.date.isin(cases.values()) & (diag.maturity == 1.0)].to_csv(TABLES / 'case_study_inputs.csv', index=False)
# ---------------------------------------------------------------- Smile vs lognormal (effect of marginals)
sl = []
for spec in ['smile', 'lognormal']:
    g = sub(spec, 'full')
    for (T, k), x in g.groupby(['maturity', 'strike_KF']):
        row = dict(spec=spec, maturity=T, KF=k, gaussian_price=x[x.copula == 'gaussian'].price.mean())
        for f in FAMS: row[f] = x[x.copula == f].diff_pct.mean()
        sl.append(row)
pd.DataFrame(sl).to_csv(TABLES / 'table_smile_vs_lognormal.csv', index=False)
json.dump(H, open(TABLES / 'headline.json', 'w'), indent=1, default=float)
print(json.dumps(H, indent=1, default=float)[:3000])


In [ ]:
# ============================================================
# Cell 10.4 - Figure 4.1 — SPX and EUR/USD spot
# ============================================================
fig, ax1 = plt.subplots(figsize=(13, 4.3))
ax1.plot(spx_daily['Date'], spx_daily['Close'], color='#1f3a5f', lw=1.0)
ax1.set_xlabel('Date'); ax1.set_ylabel('S&P 500 index level', color='#1f3a5f')
ax1.tick_params(axis='y', colors='#1f3a5f'); ax1.grid(True, alpha=0.25)
ax2 = ax1.twinx()
ax2.plot(eur_daily['Date'], eur_daily['EURUSD'], color='#8b1a1a', lw=1.0)
ax2.set_ylabel('EUR/USD spot', color='#8b1a1a')
ax2.tick_params(axis='y', colors='#8b1a1a')
plt.title('Figure 4.1. S&P 500 index and EUR/USD spot, 2018-01 to 2022-12')
plt.tight_layout()
plt.savefig(FIGURES / 'figure_4_1.png', dpi=150, bbox_inches='tight'); plt.close()
print('Figure 4.1 saved')


In [ ]:
# ============================================================
# Cell 10.5 - Figure 4.3 — Rates and SPX dividend yield
# ============================================================
fig, ax = plt.subplots(figsize=(13, 4.3))
ax.plot(rates.index, rates['r']*100,   color='#1f4e79', lw=0.9, label='USD 3M rate (r)')
ax.plot(rates.index, rates['r_f']*100, color='#a5161e', lw=0.9, label='EUR 3M Euribor (r_f)')
qcol = 'q' if 'q' in div_yield.columns else div_yield.columns[1]
ax.plot(div_yield['date'], div_yield[qcol]*100, color='#2e7d32', lw=0.9, ls='--',
        label='SPX dividend yield (q, put-call parity)')
ax.set_xlabel('Date'); ax.set_ylabel('Rate (%, annual)')
ax.axhline(0, color='black', lw=0.5, alpha=0.4)
ax.legend(loc='upper left', frameon=False, fontsize=9); ax.grid(True, alpha=0.25)
plt.title('Figure 4.3. USD and EUR 3-month rates with SPX implied dividend yield (2018-2022)')
plt.tight_layout()
plt.savefig(FIGURES / 'figure_4_3.png', dpi=150, bbox_inches='tight'); plt.close()
print('Figure 4.3 saved')


In [ ]:
# ============================================================
# Cell 10.6 - Figures 4.2 and 5.1 — EUR/USD and SPX vol surfaces (3 regime dates)
# ============================================================
dates_3  = ['2019-06-28', '2020-03-16', '2022-09-26']
labels_3 = ['Calm regime', 'COVID peak', 'EUR/USD near 20-year low']

def plot_3panel_surface(grid_df, title_prefix, fname, asset_label, palette_blue=True):
    fig = plt.figure(figsize=(16, 4.8))
    for i, (d, lbl) in enumerate(zip(dates_3, labels_3), 1):
        sub = grid_df[grid_df['date']==d]
        if len(sub) == 0:
            continue
        ax = fig.add_subplot(1, 3, i, projection='3d')
        pivg = sub.pivot_table(index='ttm_years', columns='moneyness', values='impl_vol')
        K = pivg.columns.values * 100
        T = pivg.index.values * 365
        KG, TG = np.meshgrid(K, T)
        cmap = plt.cm.Reds if 'low' in lbl.lower() or 'COVID' in lbl else plt.cm.Blues
        ax.plot_surface(KG, TG, pivg.values*100, cmap=cmap, alpha=0.85, edgecolor='none')
        ax.set_xlabel('Moneyness K/S (%)', fontsize=8)
        ax.set_ylabel('TTM (days)', fontsize=8)
        ax.set_zlabel('IV (%)', fontsize=8)
        ax.set_title(f'{asset_label} - {d}\n{lbl}', fontsize=10)
    plt.suptitle(title_prefix, fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES / fname, dpi=150, bbox_inches='tight')
    plt.close()

plot_3panel_surface(eur_grid, 'Figure 4.2. EUR/USD implied volatility surfaces across three regime dates',
                    'figure_4_2.png', 'EUR/USD')
plot_3panel_surface(spx_grid, 'Figure 5.1. S&P 500 implied volatility surfaces across three regime dates',
                    'figure_5_1.png', 'SPX')
print('Figures 4.2 and 5.1 saved')


In [ ]:
# ============================================================
# Cell 10.7 - Figure 4.4 — Returns scatter + rolling correlation
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
ax.scatter(pr['SPX_LogReturn']*100, pr['EUR_LogReturn']*100, s=8, alpha=0.45, color='#3b5b7d')
ax.axhline(0, color='black', lw=0.4, alpha=0.5); ax.axvline(0, color='black', lw=0.4, alpha=0.5)
ax.set_xlabel('SPX daily log-return (%)'); ax.set_ylabel('EUR/USD daily log-return (%)')
ax.set_title(f'SPX vs EUR/USD daily log-returns\nN={n_paired:,}  rho={pearson_full:.3f}  tau={kendall_full:.3f}', fontsize=10)
ax.grid(True, alpha=0.25)

ax = axes[1]
ax.plot(pr['Date'], pr['Roll_Corr_60d'],  color='#c8261b', lw=0.8, label='60-day rolling')
ax.plot(pr['Date'], pr['Roll_Corr_252d'], color='#1f3a5f', lw=1.0, label='252-day rolling')
ax.axhline(pearson_full, color='black', lw=0.6, ls=':', alpha=0.6, label=f'Full-sample rho={pearson_full:.3f}')
ax.set_xlabel('Date'); ax.set_ylabel('Pearson correlation')
ax.set_title('Rolling Pearson correlation between SPX and EUR/USD returns', fontsize=10)
ax.legend(loc='lower right', fontsize=8, frameon=False); ax.grid(True, alpha=0.25)

plt.suptitle('Figure 4.4. Dependence in paired daily log-returns: scatter and rolling correlation', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'figure_4_4.png', dpi=150, bbox_inches='tight'); plt.close()
print('Figure 4.4 saved')


In [ ]:
# ============================================================
# Cell 10.8 - Figure 5.2 — Empirical copula scatter
# ============================================================
u1 = rankdata(pr['SPX_LogReturn'].values) / (len(pr) + 1)
u2 = rankdata(pr['EUR_LogReturn'].values) / (len(pr) + 1)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(u1, u2, s=10, alpha=0.4, color='#5a7e5a')
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel('u_1 = empirical CDF of SPX log-return (rank pseudo-observation)')
ax.set_ylabel('u_2 = empirical CDF of EUR/USD log-return (rank pseudo-observation)')
ax.set_title(f'Figure 5.2. Empirical copula scatter (N={n_paired:,}, tau={kendall_full:.3f}, rho={pearson_full:.3f})')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURES / 'figure_5_2.png', dpi=150, bbox_inches='tight'); plt.close()
print('Figure 5.2 saved')


In [ ]:
# ============================================================
# Cell 10.9 - Figures 4.5 and 5.3-5.10
# Includes the Monte Carlo convergence runs of Cell 9.4, the pricing difference against
# standardised moneyness and the volatility versus dependence channel decomposition
# ============================================================
plt.rcParams.update({'font.family': 'serif', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
COL = {'frank': '#B8860B', 'clayton': '#B22222', 'gumbel': '#2E7D32', 'student_t': '#1F4E79'}
LAB = {'frank': 'Frank', 'clayton': 'Clayton', 'gumbel': 'Gumbel', 'student_t': 'Student-t'}
REG = ['calm', 'elevated', 'stress']; RCOL = {'calm': '#4C9F70', 'elevated': '#E0A800', 'stress': '#C0392B'}
P = pd.read_csv(TABLES / 'grid_with_zmoneyness.csv')
inv = pd.read_csv(TABLES / 'table_A1_inventory.csv'); P = P.drop(columns=['regime'], errors='ignore').merge(inv[['date', 'regime']], on='date')

# Figure 4.5 VIX with regime bands (CBOE official close)
v = pd.read_csv(INTERMED / 'VIX_daily.csv', index_col=0, parse_dates=True)['Close']['2018-01-01':'2022-12-31']
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axhspan(0, 20, color=RCOL['calm'], alpha=.12); ax.axhspan(20, 30, color=RCOL['elevated'], alpha=.12); ax.axhspan(30, 90, color=RCOL['stress'], alpha=.10)
ax.plot(v.index, v.values, color='k', lw=.8)
sd = pd.to_datetime(inv.date); ax.scatter(sd, inv.vix, s=14, c=[RCOL[r] for r in inv.regime], zorder=3, edgecolor='k', lw=.3)
ax.set_ylim(0, 90); ax.set_ylabel('VIX close'); ax.set_title('VIX daily close (Cboe) with regime bands; dots mark the 72 snapshots', fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_4_5.png', dpi=200); plt.close()

# Figure 5.3 box plots by regime (1Y, all strikes)
s = P[P.maturity == 1.0]
fig, axs = plt.subplots(1, 4, figsize=(13, 4), sharey=False)
for ax, f in zip(axs, ['frank', 'clayton', 'gumbel', 'student_t']):
    data = [s[(s.copula == f) & (s.regime == r)].diff_pct.values for r in REG]
    bp = ax.boxplot(data, tick_labels=[r.capitalize() for r in REG], patch_artist=True, widths=.6)
    for b, r in zip(bp['boxes'], REG): b.set_facecolor(RCOL[r]); b.set_alpha(.5)
    ax.axhline(0, color='grey', lw=.6, ls='--'); ax.set_title(LAB[f]); ax.set_ylabel('% difference vs Gaussian' if f == 'frank' else '')
plt.suptitle('One-year basket call: pricing difference vs Gaussian by regime (all five strikes)', fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_3.png', dpi=200); plt.close()

# Figure 5.4 time series 1Y ATM
s = P[(P.maturity == 1.0) & (P.strike_KF == 1.0)]
fig, ax = plt.subplots(figsize=(10, 3.8))
for f in ['frank', 'clayton', 'gumbel', 'student_t']:
    x = s[s.copula == f].sort_values('date'); ax.plot(pd.to_datetime(x.date), x.diff_pct, marker='o', ms=3, lw=1, color=COL[f], label=LAB[f])
ax.axhline(0, color='grey', lw=.6, ls='--')
for d in ['2020-03-16', '2022-09-26']: ax.axvline(pd.Timestamp(d), color='#C0392B', ls=':', lw=1)
ax.set_ylabel('% difference vs Gaussian'); ax.legend(frameon=False, ncol=4, fontsize=9)
ax.set_title('One-year ATM basket call (K/F = 1.00): difference vs Gaussian across the 72 snapshots', fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_4.png', dpi=200); plt.close()

# Figure 5.5 by strike, one panel per maturity
fig, axs = plt.subplots(1, 4, figsize=(13, 3.8), sharey=True)
for ax, T in zip(axs, [0.25, 0.5, 1.0, 2.0]):
    x = P[P.maturity == T].groupby(['strike_KF', 'copula']).diff_pct.mean().unstack()
    for f in ['frank', 'clayton', 'gumbel', 'student_t']: ax.plot(x.index, x[f], marker='o', color=COL[f], label=LAB[f])
    ax.axhline(0, color='grey', lw=.6); ax.axvline(1, color='grey', ls='--', lw=.6); ax.set_title(f'T = {T:g}y'); ax.set_xlabel('K / F_B')
axs[0].set_ylabel('mean % difference vs Gaussian'); axs[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_5.png', dpi=200); plt.close()

# Figure 5.6 case studies with +-1.96 SE
cs = pd.read_csv(TABLES / 'table_case_studies.csv')
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, c in zip(axs, ['calm', 'stress', 'elevated']):
    x = cs[cs.case == c]
    for f in ['frank', 'clayton', 'gumbel', 'student_t']:
        ax.errorbar(x.KF, x[f], yerr=1.96 * x[f + '_se'], marker='o', ms=4, capsize=3, color=COL[f], label=LAB[f])
    ax.axhline(0, color='grey', lw=.6); ax.set_title(f'{c.capitalize()}: {x.date.iloc[0]} (VIX {x.vix.iloc[0]:.2f})', fontsize=10); ax.set_xlabel('K / F_B')
axs[0].set_ylabel('% difference vs Gaussian (1Y)'); axs[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_8.png', dpi=200); plt.close()

# Figure 5.7 Monte Carlo convergence
try:
    c = pd.read_csv(RESULTS / 'convergence_runs.csv'); g = pd.read_csv(RESULTS / 'convergence_summary.csv')
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    x = c[(c.copula == 'gaussian') & (c.KF == 1.0)]
    for N, gg in x.groupby('N'): axs[0].scatter(np.full(len(gg), N), gg.price, s=6, color='#1F4E79', alpha=.35)
    m = g[(g.copula == 'gaussian') & (g.KF == 1.0)]
    axs[0].errorbar(m.N, m.mean_price, yerr=1.96 * m.mean_reported_se, fmt='o-', color='k', capsize=3, label='mean of 30 runs ± 1.96 × reported SE')
    axs[0].set_xscale('log'); axs[0].set_xlabel('paths N (log scale)'); axs[0].set_ylabel('price'); axs[0].legend(frameon=False, fontsize=8)
    axs[0].set_title('Gaussian, 1Y, K/F = 1.00: 30 independent runs per N', fontsize=10)
    for (f, k), mm in g.groupby(['copula', 'KF']):
        axs[1].loglog(mm.N, mm.sd_across_seeds / mm.mean_price * 100, marker='o', label=f'{LAB.get(f, f.capitalize())}, K/F={k:.2f}')
    Ns = np.array(sorted(g.N.unique())); ref = g[(g.copula == 'gaussian') & (g.KF == 1.0)].set_index('N')
    axs[1].loglog(Ns, (ref.sd_across_seeds / ref.mean_price * 100).iloc[0] * np.sqrt(Ns[0] / Ns), 'k--', lw=.8, label='1/√N reference')
    axs[1].set_xlabel('paths N'); axs[1].set_ylabel('SD across runs, % of price'); axs[1].legend(frameon=False, fontsize=8)
    axs[1].set_title('Empirical dispersion of the estimator', fontsize=10)
    plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_9.png', dpi=200); plt.close()
except FileNotFoundError:
    print('convergence not ready')

# Figure 5.8 SE distribution
m = P; rel = (m.se / m.price * 100).replace([np.inf], np.nan).dropna()
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
axs[0].hist(m.se, bins=60, color='#1F4E79'); axs[0].set_title('absolute SE of price', fontsize=10)
axs[1].hist(rel.clip(upper=rel.quantile(.99)), bins=60, color='#1F4E79'); axs[1].set_title('relative SE, % (clipped at P99)', fontsize=10)
dd = m[m.copula != 'gaussian']; axs[2].hist(dd.se_diff_pct.clip(upper=dd.se_diff_pct.quantile(.99)), bins=60, color='#2E7D32'); axs[2].set_title('SE of % difference vs Gaussian (P99 clip)', fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_10.png', dpi=200); plt.close()

# Figure 5.9 standardised moneyness
s = P[(P.copula == 'gumbel') & (P.strike_KF >= 1.0)]
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for r in REG:
    x = s[s.regime == r]; axs[0].scatter(x.z_moneyness, x.diff_pct, s=10, alpha=.6, color=RCOL[r], label=r.capitalize())
axs[0].set_xlabel('standardised moneyness  ln(K/F_B) / (σ_B √T)'); axs[0].set_ylabel('Gumbel % difference vs Gaussian'); axs[0].legend(frameon=False)
axs[0].set_title('Gumbel vs Gaussian against strike distance in volatility units', fontsize=10)
s2 = P[(P.copula == 'frank') & (P.strike_KF >= 1.0)]
for r in REG:
    x = s2[s2.regime == r]; axs[1].scatter(x.z_moneyness, x.diff_pct, s=10, alpha=.6, color=RCOL[r], label=r.capitalize())
axs[1].set_xlabel('standardised moneyness'); axs[1].set_ylabel('Frank % difference vs Gaussian'); axs[1].set_title('Frank vs Gaussian', fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_6.png', dpi=200); plt.close()

# Figure 5.10 decomposition heatmap
dec = pd.read_csv(TABLES / 'table_5_5_decomposition_1Y_110.csv')
M = dec.pivot(index='snapshot_regime', columns='copula_params', values='gumbel_minus_frank').loc[REG, ['calm', 'elevated', 'stress', 'full']]
fig, ax = plt.subplots(figsize=(6.5, 3.6)); im = ax.imshow(M.values, cmap='Greens')
ax.set_xticks(range(4)); ax.set_xticklabels(['calm\ncopula', 'elevated\ncopula', 'stress\ncopula', 'full-sample\ncopula'])
ax.set_yticks(range(3)); ax.set_yticklabels(['calm snapshots', 'elevated snapshots', 'stress snapshots'])
for i in range(3):
    for j in range(4): ax.text(j, i, f'{M.values[i, j]:.1f}', ha='center', va='center', fontsize=10)
ax.set_title('Gumbel minus Frank, 1Y, K/F = 1.10 (percentage points)', fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / 'figure_5_7.png', dpi=200); plt.close()
print('figures done')


In [ ]:
# ============================================================
# Cell 10.10 - results_full.json: every number reported in the thesis
# ============================================================
import sys, scipy
H = json.load(open(TABLES / 'headline.json'))
tbl = lambda name: pd.read_csv(TABLES / name).to_dict(orient='records')
results_full = {
    'metadata': {'author': 'Mariglen Tanushi', 'institution': 'Universita Cattolica del Sacro Cuore, Milano',
                 'supervisor': 'Prof. Paolo Gualtieri', 'pipeline_version': '1.0',
                 'pipeline_run_at': datetime.now(timezone.utc).isoformat(),
                 # SVI fits are non-convex: record the library versions that produced these numbers
                 'environment': {'python': sys.version.split()[0], 'numpy': np.__version__, 'scipy': scipy.__version__,
                                 'pandas': pd.__version__}},
    'methodology': {
        'marginals': 'option-implied: raw-SVI per pillar (butterfly/calendar penalties), total-variance interpolation in T, '
                     'Breeden-Litzenberger CDF, inverse CDF; flat-smile lognormal at ATM-forward vol as benchmark',
        'spx_2y': 'extrapolated beyond the 1.38y pillar (flat ATM vol, fixed smile in standardised moneyness)',
        'copulas_priced': list(globals().get('PRICING_FAMILIES', ['gaussian', 'frank', 'clayton', 'gumbel', 'student_t'])), 'antithetic': 'applied to driving uniforms', 'common_random_numbers': True,
        'strike': 'K = (K/F_B) * F_B, F_B = 0.5 e^{(r-q)T} + 0.5 e^{(r-r_f)T}', 'paths_per_cell': 100000,
        'parameter_sets': ['full', 'calm', 'elevated', 'stress'],
        'regime_snapshots': 'Cboe VIX close on snapshot date; calm<20<=elevated<30<=stress',
        'regime_returns': 'previous trading day Cboe VIX close',
        'euribor': 'latest monthly average complete on the snapshot date (ECB FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA)'},
    'paired_returns_full_sample': paired_stats,
    'calibration': calib, 'robustness': robust, 'headline': H,
    'table_5_2_atm': tbl('table_5_2_atm.csv'), 'table_5_3_grid': tbl('table_5_3_grid.csv'),
    'table_5_4_regime_strike_full': tbl('table_5_4_regime_strike_full.csv'),
    'table_5_4b_matched': tbl('table_5_4b_regime_strike_matched.csv'),
    'table_5_5_decomposition': tbl('table_5_5_decomposition_1Y_110.csv'),
    'case_studies': tbl('table_case_studies.csv'),
    'convergence': pd.read_csv(RESULTS / 'convergence_summary.csv').to_dict(orient='records'),
    'copula_extra_checks': (json.load(open(RESULTS / 'copula_extra_checks.json'))
                            if (RESULTS / 'copula_extra_checks.json').exists() else None),
    'measure_check': (pd.read_csv(RESULTS / 'table_qmeasure.csv').to_dict(orient='records')
                      if (RESULTS / 'table_qmeasure.csv').exists() else None),
}
with open(OUTPUTS / 'results_full.json', 'w') as f:
    json.dump(results_full, f, indent=2, default=float)
print(f'Wrote {OUTPUTS / "results_full.json"}  ({(OUTPUTS / "results_full.json").stat().st_size:,} bytes)')
